# Data Collection

This notebook goes through the steps taken to web scrape financial documents and data from JSE and BSE's websites. It also goes through the steps taken for direct text extraction and Cloud OCR.

**Note that you will need to apply for your own Cloud OCR key, it is free and can be accessed from https://ocr.space/ocrapi/freekey.

## JSE (Jamaica Stock Exchange) Data Scraper

In [1]:
# Import the required packages
import io
import json
import logging
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Optional
import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup

# Set the configurations for JSE scraping
BASE_URL = "https://www.jamstockex.com"
WP_API_POSTS = f"{BASE_URL}/wp-json/wp/v2/posts"
NEWSROOM_URL = f"{BASE_URL}/newsroom/"
DAILY_QUOTE_PDF_URL = f"{BASE_URL}/trading/trade-quotes/daily-quote-pdf/"
REQUEST_DELAY_SECONDS = 1.5
TIMEOUT = 20

logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(message)s",)

log = logging.getLogger("jse_scraper")

# Create the JSE session header
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.jamstockex.com/",})

In [2]:
# Define a function to clean the scraped text
def clean_html_to_text(html: str) -> str:
    """Strip HTML tags/scripts and collapse whitespace into plain text."""
    soup = BeautifulSoup(html or "", "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    return " ".join(text.split())

In [3]:
# Define function to retrieve data from the WordPress API
def try_wp_api(max_pages: int, after_date: str = None) -> list:
    
    records = []
    page = 1
    per_page = 20
    base_params = {"per_page": per_page, "_embed": "true"}
    if after_date:
        base_params["after"] = f"{after_date}T00:00:00"

    # Lightweight probe request to check the API even responds.
    try:
        probe = session.get(WP_API_POSTS, params = {**base_params, "per_page": 1, "page": 1}, timeout = TIMEOUT)
    except requests.RequestException as e:
        log.warning("WP API probe failed (network error): %s", e)
        return []

    if probe.status_code != 200:
        log.warning("WP API probe returned status %s -- will use HTML fallback.", probe.status_code)
        return []

    try:
        probe.json()
    except ValueError:
        log.warning("WP API probe did not return JSON -- will use HTML fallback.")
        return []

    total_posts = probe.headers.get("X-WP-Total")
    log.info("WP API is reachable. Total posts matching filter: %s", total_posts)

    while page <= max_pages:
        params = {**base_params, "page": page}
        try:
            resp = session.get(WP_API_POSTS, params=params, timeout=TIMEOUT)
        except requests.RequestException as e:
            log.warning("Request error on API page %s: %s", page, e)
            break

        if resp.status_code == 400:
            log.info("Reached end of available pages at page %s.", page)
            break
        if resp.status_code != 200:
            log.warning("API page %s returned status %s -- stopping API pull.", page, resp.status_code)
            break

        posts = resp.json()
        if not posts:
            break

        for post in posts:
            title = clean_html_to_text(post.get("title", {}).get("rendered", ""))
            body = clean_html_to_text(post.get("content", {}).get("rendered", ""))
            categories = []
            embedded = post.get("_embedded", {})
            for term_group in embedded.get("wp:term", []):
                for term in term_group:
                    if term.get("taxonomy") in ("category", "post_tag"):
                        categories.append(term.get("name"))

            records.append({
                "id": f"wp-{post.get('id')}",
                "source": "jse",
                "title": title,
                "date": post.get("date"),
                "url": post.get("link"),
                "categories": categories,
                "text": body,
                "retrieved_via": "wp_api",
            })

        log.info("API page %s: collected %s posts (running total: %s)", page, len(posts), len(records))
        page += 1
        time.sleep(REQUEST_DELAY_SECONDS)

    return records

In [4]:
# Define a function to scrape the HTML if the API scraping fails
def scrape_article(url: str) -> Optional[dict]:
  
    try:
        resp = session.get(url, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to fetch article %s: %s", url, e)
        return None

    if resp.status_code != 200:
        log.warning("Article %s returned status %s.", url, resp.status_code)
        return None

    soup = BeautifulSoup(resp.text, "html.parser")

    title_tag = soup.find(["h1", "h2"], class_=lambda c: c and "title" in c.lower()) or soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else url

    body_tag = (
        soup.find("div", class_=lambda c: c and "entry-content" in c.lower())
        or soup.find("article")
        or soup.find("main"))
    
    body_text = clean_html_to_text(str(body_tag)) if body_tag else clean_html_to_text(resp.text)

    date_tag = soup.find("time")
    date_val = date_tag.get("datetime") if date_tag and date_tag.get("datetime") else (
        date_tag.get_text(strip=True) if date_tag else None)

    return {
        "id": f"html-{abs(hash(url))}",
        "source": "jse",
        "title": title,
        "date": date_val,
        "url": url,
        "categories": [],
        "text": body_text,
        "retrieved_via": "html_fallback",}

# Define a function to failover to the HTML scraper if the API fails
def html_fallback_scrape(max_pages: int) -> list:
    
    records = []
    seen_urls = set()

    for page in range(1, max_pages + 1):
        listing_url = NEWSROOM_URL if page == 1 else f"{NEWSROOM_URL}page/{page}/"
        try:
            resp = session.get(listing_url, timeout = TIMEOUT)
        except requests.RequestException as e:
            log.warning("Failed to fetch listing page %s: %s", listing_url, e)
            break

        if resp.status_code == 404:
            log.info("No more listing pages (404 at page %s).", page)
            break
        if resp.status_code != 200:
            log.warning("Listing page %s returned status %s.", listing_url, resp.status_code)
            break

        soup = BeautifulSoup(resp.text, "html.parser")

        article_links = set()
        for a in soup.select("article a[href]"):
            href = a.get("href", "")
            if href.startswith(BASE_URL) and href not in seen_urls:
                article_links.add(href)

        if not article_links:
            for a in soup.select("a[href]"):
                href = a.get("href", "")
                if href.startswith(BASE_URL) and href.rstrip("/").count("/") >= 3:
                    article_links.add(href)

        if not article_links:
            log.info("No article links found on page %s -- stopping.", page)
            break

        log.info("Listing page %s: found %s candidate article links.", page, len(article_links))

        for url in article_links:
            if url in seen_urls:
                continue
            seen_urls.add(url)
            record = scrape_article(url)
            if record:
                records.append(record)
            time.sleep(REQUEST_DELAY_SECONDS)

        time.sleep(REQUEST_DELAY_SECONDS)

    return records

## JSE Financial Data Collection

In [6]:
# Set the cut off date (2 years) and max number of web pages to scrape
cutoff_date = (date.today() - timedelta(days = 365*2)).isoformat()
MAX_PAGES = 280 

# Scrape the website and save the number of records
records = try_wp_api(max_pages = MAX_PAGES, after_date = cutoff_date)

# If there are no records, return a message and try the HTML scraper
if not records:
    print("WP API returned nothing -- using HTML fallback.")
    records = html_fallback_scrape(max_pages = MAX_PAGES)

# Print the number of records collected
print(f"Total records collected: {len(records)}")

2026-08-16 21:28:55,488 [INFO] WP API is reachable. Total posts matching filter: 5531
2026-08-16 21:28:58,110 [INFO] API page 1: collected 20 posts (running total: 20)
2026-08-16 21:29:02,252 [INFO] API page 2: collected 20 posts (running total: 40)
2026-08-16 21:29:07,540 [INFO] API page 3: collected 20 posts (running total: 60)
2026-08-16 21:29:12,718 [INFO] API page 4: collected 20 posts (running total: 80)
2026-08-16 21:29:17,882 [INFO] API page 5: collected 20 posts (running total: 100)
2026-08-16 21:29:23,584 [INFO] API page 6: collected 20 posts (running total: 120)
2026-08-16 21:29:28,575 [INFO] API page 7: collected 20 posts (running total: 140)
2026-08-16 21:29:33,790 [INFO] API page 8: collected 20 posts (running total: 160)
2026-08-16 21:29:38,917 [INFO] API page 9: collected 20 posts (running total: 180)
2026-08-16 21:29:44,186 [INFO] API page 10: collected 20 posts (running total: 200)
2026-08-16 21:29:49,263 [INFO] API page 11: collected 20 posts (running total: 220)
202

2026-08-16 21:37:18,115 [INFO] API page 98: collected 20 posts (running total: 1960)
2026-08-16 21:37:23,290 [INFO] API page 99: collected 20 posts (running total: 1980)
2026-08-16 21:37:28,765 [INFO] API page 100: collected 20 posts (running total: 2000)
2026-08-16 21:37:33,690 [INFO] API page 101: collected 20 posts (running total: 2020)
2026-08-16 21:37:38,650 [INFO] API page 102: collected 20 posts (running total: 2040)
2026-08-16 21:37:43,618 [INFO] API page 103: collected 20 posts (running total: 2060)
2026-08-16 21:37:48,663 [INFO] API page 104: collected 20 posts (running total: 2080)
2026-08-16 21:37:53,852 [INFO] API page 105: collected 20 posts (running total: 2100)
2026-08-16 21:37:59,222 [INFO] API page 106: collected 20 posts (running total: 2120)
2026-08-16 21:38:04,309 [INFO] API page 107: collected 20 posts (running total: 2140)
2026-08-16 21:38:09,402 [INFO] API page 108: collected 20 posts (running total: 2160)
2026-08-16 21:38:14,446 [INFO] API page 109: collected 2

2026-08-16 21:45:37,503 [INFO] API page 194: collected 20 posts (running total: 3880)
2026-08-16 21:45:42,716 [INFO] API page 195: collected 20 posts (running total: 3900)
2026-08-16 21:45:47,955 [INFO] API page 196: collected 20 posts (running total: 3920)
2026-08-16 21:45:52,921 [INFO] API page 197: collected 20 posts (running total: 3940)
2026-08-16 21:45:58,244 [INFO] API page 198: collected 20 posts (running total: 3960)
2026-08-16 21:46:03,202 [INFO] API page 199: collected 20 posts (running total: 3980)
2026-08-16 21:46:08,481 [INFO] API page 200: collected 20 posts (running total: 4000)
2026-08-16 21:46:13,650 [INFO] API page 201: collected 20 posts (running total: 4020)
2026-08-16 21:46:18,837 [INFO] API page 202: collected 20 posts (running total: 4040)
2026-08-16 21:46:24,179 [INFO] API page 203: collected 20 posts (running total: 4060)
2026-08-16 21:46:29,641 [INFO] API page 204: collected 20 posts (running total: 4080)
2026-08-16 21:46:34,750 [INFO] API page 205: collected

Total records collected: 5531


In [7]:
# Set the output path
out_path = Path("jse_corpus.json")

# Save the extracted records
out_path.write_text(json.dumps(records, indent = 2, ensure_ascii = False), encoding = "utf-8")

# Print number of records saved
print(f"Saved {len(records)} records to {out_path.resolve()}")

Saved 5531 records to /home/areas001/Final Project/Final Notebooks/jse_corpus.json


In [8]:
# Save the scraped data in a DataFrame
df = pd.DataFrame(records)
print(df.shape)

# Preview the first 10 rows for a sanity check
df[["title", "date", "url", "retrieved_via"]].head(10)

(5531, 8)


,title,date,url,retrieved_via
0,Tropical Battery Company Limited (TROPICAL) – ...,2026-08-14T19:43:27,https://www.jamstockex.com/tropical-battery-co...,wp_api
1,PROVEN GROUP LIMITED UNAUDITED FIRST QUARTER R...,2026-08-14T19:43:05,https://www.jamstockex.com/proven-group-limite...,wp_api
2,Dolphin Cove Limited (DCOVE) – Unaudited Finan...,2026-08-14T19:36:32,https://www.jamstockex.com/dolphin-cove-limite...,wp_api
3,Kingston Properties Limited (KPREIT) Unaudited...,2026-08-14T19:31:17,https://www.jamstockex.com/kingston-properties...,wp_api
4,Eppley Caribbean Property Fund Limited SCC (CP...,2026-08-14T19:29:50,https://www.jamstockex.com/eppley-caribbean-pr...,wp_api
5,Eppley Caribbean Property Fund Limited SCC (CP...,2026-08-14T19:29:13,https://www.jamstockex.com/eppley-caribbean-pr...,wp_api
6,Eppley Caribbean Property Fund Limited SCC – V...,2026-08-14T19:28:39,https://www.jamstockex.com/eppley-caribbean-pr...,wp_api
7,FosRich Company Limited (FOSRICH) Unaudited Fi...,2026-08-14T19:26:55,https://www.jamstockex.com/fosrich-company-lim...,wp_api
8,Consolidated Bakeries Jamaica Limited (PURITY)...,2026-08-14T19:25:29,https://www.jamstockex.com/consolidated-bakeri...,wp_api
9,Carreras Limited (CAR) Unaudited Financial Sta...,2026-08-14T19:18:10,https://www.jamstockex.com/carreras-limited-ca...,wp_api


## JSE Price Data Collection

## 10. Price-scraper configuration

In [3]:
# list all JSE's market codes
MARKET_CODES = {
    "main": 31,
    "junior": 22,
    "combined": 50,
    "usd": 49,
    "bond": 51,
    "usd_bond": 52,
    "jmd_government_bond": 71,}

# Specify the trade data columns
QUOTE_COLUMNS = [
    "symbol", "last_traded_price", "closing_price", "price_change",
    "closing_bid", "closing_ask", "volume", "today_high", "today_low",
    "52wk_high", "52wk_low", "total_prev_yr_div", "total_current_yr_div",]

In [10]:
# Define a function to extract the company symbols
def _looks_like_symbol(cell: str) -> bool:
    
    if not cell:
        return False
    cell = cell.strip()
    if not cell or len(cell) > 15:
        return False
    if cell.lower() in ("symbol", "ordinary shares", "preference shares"):
        return False
    return any(c.isalpha() for c in cell) and cell.upper() == cell

# Define the function to fetch the daily trade reports
def fetch_daily_quotes(quote_date: date, market_code: int) -> list:
    
    params = {"date": quote_date.isoformat(), "market": market_code}
    try:
        resp = session.get(DAILY_QUOTE_PDF_URL, params=params, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Request failed for %s market=%s: %s", quote_date, market_code, e)
        return []

    if resp.status_code != 200:
        log.info("No PDF for %s market=%s (status %s) -- likely non-trading day.",
                  quote_date, market_code, resp.status_code)
        return []

    content_type = resp.headers.get("Content-Type", "")
    if "pdf" not in content_type.lower():
        log.info("Response for %s market=%s wasn't a PDF (Content-Type: %s) -- skipping.",
                  quote_date, market_code, content_type)
        return []

    rows = []
    try:
        with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
            for page in pdf.pages:
                tables = page.extract_tables()
                for table in tables:
                    for raw_row in table:
                        if not raw_row:
                            continue
                        cleaned = [c.strip() if c else "" for c in raw_row]
                        if not cleaned or not _looks_like_symbol(cleaned[0]):
                            continue
                        cleaned = (cleaned + [""] * len(QUOTE_COLUMNS))[:len(QUOTE_COLUMNS)]
                        row_dict = dict(zip(QUOTE_COLUMNS, cleaned))
                        row_dict["date"] = quote_date.isoformat()
                        row_dict["market"] = market_code
                        rows.append(row_dict)
    except Exception as e:
        log.warning("Failed to parse PDF for %s market=%s: %s", quote_date, market_code, e)
        return []

    return rows

In [11]:
# Perform a sanity check to ensure the columns match
test_date = date(2026, 4, 14)
test_rows = fetch_daily_quotes(test_date, MARKET_CODES["main"])

# Print the sample of rows
print(f"Extracted {len(test_rows)} rows for {test_date}")
test_rows[:5]

Extracted 34 rows for 2026-04-14


[{'symbol': 'ASBH',
  'last_traded_price': '26.00',
  'closing_price': '26.18',
  'price_change': '-0.68',
  'closing_bid': '26.75',
  'closing_ask': '29.46',
  'volume': '2,860',
  'today_high': '27.01',
  'today_low': '26.00',
  '52wk_high': '40.00',
  '52wk_low': '20.00',
  'total_prev_yr_div': '0.0397',
  'total_current_yr_div': '0.0132',
  'date': '2026-04-14',
  'market': 31},
 {'symbol': 'BRG',
  'last_traded_price': '5.15',
  'closing_price': '5.15',
  'price_change': '-0.23',
  'closing_bid': '5.15',
  'closing_ask': '5.70',
  'volume': '979',
  'today_high': '5.15',
  'today_low': '5.15',
  '52wk_high': '7.89',
  '52wk_low': '5.00',
  'total_prev_yr_div': '',
  'total_current_yr_div': '',
  'date': '2026-04-14',
  'market': 31},
 {'symbol': 'CPJ',
  'last_traded_price': '5.68',
  'closing_price': '5.67',
  'price_change': '-0.06',
  'closing_bid': '5.55',
  'closing_ask': '5.70',
  'volume': '1,161',
  'today_high': '5.68',
  'today_low': '5.55',
  '52wk_high': '9.05',
  '52w

In [12]:
# Set the start and end dates
START_DATE = date.today() - timedelta(days = 365 * 2)
END_DATE = date.today()

# Create the variable to save the price rows and set the number days
all_price_rows = []
current = START_DATE
total_days = (END_DATE - START_DATE).days + 1
processed = 0

# For the date period, scrape the price rows for each market
while current <= END_DATE:
    for market_name, market_code in MARKET_CODES.items():
        rows = fetch_daily_quotes(current, market_code)
        for r in rows:
            r["market_name"] = market_name
        all_price_rows.extend(rows)
        time.sleep(REQUEST_DELAY_SECONDS)

    processed += 1
    if processed % 25 == 0:
        log.info("Processed %s/%s days -- %s rows collected so far.",
                  processed, total_days, len(all_price_rows))

    current += timedelta(days=1)

# Print once completed
print(f"Done. Total rows collected: {len(all_price_rows)}")

2026-08-16 22:01:44,933 [INFO] Processed 25/731 days -- 2339 rows collected so far.
2026-08-16 22:10:36,064 [INFO] Processed 50/731 days -- 4961 rows collected so far.
2026-08-16 22:19:07,477 [INFO] Processed 75/731 days -- 7154 rows collected so far.
2026-08-16 22:27:50,964 [INFO] Processed 100/731 days -- 9620 rows collected so far.
2026-08-16 22:36:31,544 [INFO] Processed 125/731 days -- 12086 rows collected so far.
2026-08-16 22:44:48,892 [INFO] Processed 150/731 days -- 14004 rows collected so far.
2026-08-16 22:53:32,236 [INFO] Processed 175/731 days -- 16589 rows collected so far.
2026-08-16 23:02:24,478 [INFO] Processed 200/731 days -- 18881 rows collected so far.
2026-08-16 23:11:22,144 [INFO] Processed 225/731 days -- 21289 rows collected so far.
2026-08-16 23:19:48,427 [INFO] Processed 250/731 days -- 23327 rows collected so far.
2026-08-16 23:28:29,881 [INFO] Processed 275/731 days -- 25793 rows collected so far.
2026-08-16 23:37:09,586 [INFO] Processed 300/731 days -- 2812

Done. Total rows collected: 69506


In [13]:
# Save the price data to a DataFrame
price_df = pd.DataFrame(all_price_rows)

# Save the price data to csv
price_df.to_csv("jse_price_data.csv", index = False)

# Preview the first 10 rows as a sanity check
print(f"Saved {len(price_df)} rows to jse_price_data.csv")
price_df.head(10)

Saved 69506 rows to jse_price_data.csv


,symbol,last_traded_price,closing_price,price_change,closing_bid,closing_ask,volume,today_high,today_low,52wk_high,52wk_low,total_prev_yr_div,total_current_yr_div,date,market,market_name
0,ASBH,37.50,33.95,-5.03,35.00,37.49,474,37.50,33.50,45.00,28.01,,,2024-08-16,31,main
1,BRG,6.36,6.85,0.05,6.50,6.85,"4,157",6.85,6.36,11.98,3.90,,,2024-08-16,31,main
2,CPJ,9.03,9.02,0.00,9.03,9.24,"33,885",9.03,9.01,13.89,8.00,,,2024-08-16,31,main
3,CPFV,40.75,40.75,3.21,35.01,40.75,84,40.75,40.75,42.10,30.00,0.0288,0.0302,2024-08-16,31,main
4,FIRSTROCKJMD,10.30,10.36,0.06,10.30,10.92,"44,202",10.93,10.30,11.00,6.00,,,2024-08-16,31,main
5,GK,73.40,74.02,-0.10,73.01,74.22,"23,166",74.22,73.01,82.90,60.00,2.1700,1.6200,2024-08-16,31,main
6,ENERGY,0.82,0.82,-0.06,0.82,0.87,"4,853",0.88,0.82,1.70,0.65,,,2024-08-16,31,main
7,JP,22.95,22.86,0.00,21.50,22.95,0,0.00,0.00,27.45,18.00,0.3000,0.3000,2024-08-16,31,main
8,JMMBGL,21.90,22.16,-0.41,22.78,22.79,"51,609",23.03,21.90,30.00,21.90,0.2500,0.2500,2024-08-16,31,main
9,KPREIT,9.00,9.00,0.00,9.00,9.18,"10,000",9.00,9.00,9.65,6.20,0.0015,0.0006,2024-08-16,31,main


## BSE Data Scraper

In [4]:
# Set the BSE configuration for web scrapping
BASE_URL = "https://bse.com.bb"
WP_API_POSTS = f"{BASE_URL}/wp-json/wp/v2/posts"
NEWS_URL = f"{BASE_URL}/news/"

log = logging.getLogger("bse_scraper")

# Set the BSE session header
session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://bse.com.bb/",})

In [18]:
# Test on 2 pages to ensure that the scraper is successful
test_records = try_wp_api(max_pages = 2, after_date = cutoff_date)

# If the test fails, try the HTML scraper
if not test_records:
    print("WP API returned nothing -- trying HTML fallback...")
    test_records = html_fallback_scrape(max_pages = 2)

# Print the number of rows collected
print(f"Collected {len(test_records)} records in test run.")
if test_records:
    {k: v for k, v in test_records[0].items() if k != "content_html"}

Cutoff date (2 years back): 2024-08-17


2026-08-17 02:08:27,379 [INFO] WP API is reachable. Total posts matching filter: 604
2026-08-17 02:08:28,347 [INFO] API page 1: collected 20 posts (running total: 20)
2026-08-17 02:08:30,913 [INFO] API page 2: collected 20 posts (running total: 40)


Collected 40 records in test run.


In [19]:
# Run the scrapper on the full number of pages
MAX_PAGES = 620 

# Save the scrapped data
records = try_wp_api(max_pages = MAX_PAGES, after_date = cutoff_date)

# If there were no records, try the HTML scraper
if not records:
    print("WP API returned nothing -- using HTML fallback.")
    records = html_fallback_scrape(max_pages = MAX_PAGES)

# Print the total number of records collected
print(f"Total records collected: {len(records)}")

2026-08-17 02:08:32,642 [INFO] WP API is reachable. Total posts matching filter: 604
2026-08-17 02:08:33,596 [INFO] API page 1: collected 20 posts (running total: 20)
2026-08-17 02:08:36,230 [INFO] API page 2: collected 20 posts (running total: 40)
2026-08-17 02:08:39,054 [INFO] API page 3: collected 20 posts (running total: 60)
2026-08-17 02:08:41,712 [INFO] API page 4: collected 20 posts (running total: 80)
2026-08-17 02:08:44,248 [INFO] API page 5: collected 20 posts (running total: 100)
2026-08-17 02:08:46,560 [INFO] API page 6: collected 20 posts (running total: 120)
2026-08-17 02:08:49,139 [INFO] API page 7: collected 20 posts (running total: 140)
2026-08-17 02:08:51,755 [INFO] API page 8: collected 20 posts (running total: 160)
2026-08-17 02:08:54,369 [INFO] API page 9: collected 20 posts (running total: 180)
2026-08-17 02:08:56,759 [INFO] API page 10: collected 20 posts (running total: 200)
2026-08-17 02:08:59,294 [INFO] API page 11: collected 20 posts (running total: 220)
2026

Total records collected: 604


In [20]:
# Drop the raw content_html from the saved corpus 
disclosures_out = [{k: v for k, v in r.items() if k != "content_html"} for r in records]

# Save the BSE scrapped data
out_path = Path("bse_corpus.json")
out_path.write_text(json.dumps(disclosures_out, indent = 2, ensure_ascii = False), encoding = "utf-8")

# Print the number of records collected
print(f"Saved {len(disclosures_out)} records to {out_path.resolve()}")

Saved 604 records to /home/areas001/Final Project/Final Notebooks/bse_corpus.json


In [21]:
# Save the data in a DataFrame
df = pd.DataFrame(disclosures_out)

# Print the DataFrame shape
print(df.shape)

# Preview the first 10 rows of the BSE data
df[["title", "date", "url", "retrieved_via"]].head(10)

(604, 8)


,title,date,url,retrieved_via
0,"BSE Trading Report Package August 14, 2026",2026-08-14T15:21:35,https://bse.com.bb/news/30043,wp_api
1,"BSE Trading Report Package August 13, 2026",2026-08-13T14:29:03,https://bse.com.bb/news/30038,wp_api
2,"BSE Trading Report Package August 12, 2026",2026-08-12T14:13:30,https://bse.com.bb/news/30011,wp_api
3,"BSE Trading Report Package August 11, 2026",2026-08-11T14:12:13,https://bse.com.bb/news/30002,wp_api
4,"BSE Trading Report Package August 10, 2026",2026-08-10T14:35:58,https://bse.com.bb/news/29997,wp_api
5,"BSE Trading Report Package August 7, 2026",2026-08-07T14:49:24,https://bse.com.bb/news/29988,wp_api
6,"BSE Trading Report Package August 6, 2026",2026-08-06T14:20:42,https://bse.com.bb/news/29982,wp_api
7,"BSE Trading Report Package August 5, 2026",2026-08-05T14:22:08,https://bse.com.bb/news/29975,wp_api
8,"BSE Trading Report Package August 4, 2026",2026-08-04T14:09:19,https://bse.com.bb/news/29969,wp_api
9,"BSE Trading Report Package July 31, 2026",2026-07-31T14:42:58,https://bse.com.bb/news/29964,wp_api


## BSE Price Data Collection

In [22]:
# Import the required package
import re

# Define a function to clean the BSE post links
def clean_url(href: str) -> str:
    href = href.strip()
    href = re.sub(r'^https?:/+\s*(https?://)', r'\1', href)
    return href

# Define a function to download the trade PDFs from the BSE links
def extract_report_links(content_html: str) -> dict:

    soup = BeautifulSoup(content_html or "", "html.parser")
    links = {"trading_report_pdf": None, "share_summary_xls": None, "market_cap_pdf": None}
    for a in soup.find_all("a", href=True):
        href = clean_url(a["href"])
        text = a.get_text(strip=True).lower()
        combined = f"{text} {href.lower()}"

        if "market cap" in combined or "market-cap" in combined:
            links["market_cap_pdf"] = href
        elif "share summary" in combined or "share-summary" in combined:
            links["share_summary_xls"] = href
        elif "trading report" in combined or href.lower().endswith(".pdf"):
            links["trading_report_pdf"] = href

    return links


# Save the extracted price rows
report_posts = []
for r in records:
    if "trading report package" in r["title"].lower():
        links = extract_report_links(r["content_html"])
        report_posts.append({
            "date": r["date"][:10] if r["date"] else None,
            "title": r["title"],
            "url": r["url"],
            **links,})

# Print the number of trade posts found
print(f"Found {len(report_posts)} Trading Report Package posts.")

# Preview the first 3 as a sanity check
report_posts[:3]

Found 495 Trading Report Package posts.


[{'date': '2026-08-14',
  'title': 'BSE Trading Report Package August 14, 2026',
  'url': 'https://bse.com.bb/news/30043',
  'trading_report_pdf': 'http://bse.com.bb/wp-content/uploads/2026/08/20260814.pdf',
  'share_summary_xls': 'http://bse.com.bb/wp-content/uploads/2026/08/Share-Summary-August-14-2026.xls',
  'market_cap_pdf': 'http://bse.com.bb/wp-content/uploads/2026/08/Market-Cap-August-14-2026.pdf'},
 {'date': '2026-08-13',
  'title': 'BSE Trading Report Package August 13, 2026',
  'url': 'https://bse.com.bb/news/30038',
  'trading_report_pdf': 'http://bse.com.bb/wp-content/uploads/2026/08/20260813.pdf',
  'share_summary_xls': 'http://bse.com.bb/wp-content/uploads/2026/08/Share-Summary-August-13-2026.xls',
  'market_cap_pdf': 'http://bse.com.bb/wp-content/uploads/2026/08/Market-Cap-August-13-2026.pdf'},
 {'date': '2026-08-12',
  'title': 'BSE Trading Report Package August 12, 2026',
  'url': 'https://bse.com.bb/news/30011',
  'trading_report_pdf': 'http://bse.com.bb/wp-content/u

In [23]:
# Map the column names to the extracted column names
KNOWN_HEADER_MAP = {
    "last trade date": "last_trade_date",
    "volume": "volume",
    "high": "day_high",
    "low": "day_low",
    "last": "last_traded_price",
    "last  close": "last_traded_price",
    "close": "closing_price",
    "current close": "current_close",
    "price change": "price_change",
    "bid price": "bid_price",
    "ask price": "ask_price",
    "bid          size": "bid_size",
    "ask          size": "ask_size",
    "bid size": "bid_size",
    "ask size": "ask_size",}

# Define the function to extract the header row
def find_header_row(raw_df, max_scan_rows = 10):

    for i in range(min(max_scan_rows, len(raw_df))):
        row_values = [str(c).strip().lower() if pd.notna(c) else "" for c in raw_df.iloc[i]]
        if "volume" in row_values or "last trade date" in row_values:
            return i
    return None

# Extract the trade data from the downloaded xls files
def parse_share_summary_xls(xls_url: str, report_date: str) -> list:
 
    try:
        resp = session.get(xls_url, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to fetch %s: %s", xls_url, e)
        return []

    if resp.status_code != 200:
        log.warning("%s returned status %s.", xls_url, resp.status_code)
        return []

    try:
        raw_df = pd.read_excel(io.BytesIO(resp.content), header=None)
    except Exception as e:
        log.warning("Failed to parse %s: %s", xls_url, e)
        return []

    header_idx = find_header_row(raw_df)
    if header_idx is None:
        log.warning("Could not find a recognizable header row in %s (date=%s) -- skipping. "
                    "This file may use a different report layout.", xls_url, report_date)
        return []

    header_row = raw_df.iloc[header_idx]
    data_df = raw_df.iloc[header_idx + 1:].reset_index(drop=True)

    new_columns = []
    for i, c in enumerate(header_row):
        if i == 0:
            new_columns.append("company_name")
            continue
        label = str(c).strip().lower() if pd.notna(c) else ""
        new_columns.append(KNOWN_HEADER_MAP.get(label, f"unrecognized_{label or f'col_{i}'}"))
    data_df.columns = new_columns

    data_df = data_df.dropna(subset=["company_name"])

    unrecognized = [c for c in data_df.columns if c.startswith("unrecognized_")]
    if unrecognized:
        log.warning("File %s (date=%s) has unrecognized columns: %s", xls_url, report_date, unrecognized)

    data_df["date"] = report_date
    return data_df.to_dict(orient="records")


# Run a test on the most recent report as a sanity check
test_report = next((r for r in report_posts if r["share_summary_xls"]), None)
if test_report:
    test_price_rows = parse_share_summary_xls(test_report["share_summary_xls"], test_report["date"])
    report_date = test_report["date"]
    print(f"Extracted {len(test_price_rows)} rows for {report_date}")
    test_price_rows[:5]
else:
    print("No Share Summary .xls link found.")

Extracted 29 rows for 2026-08-14


In [24]:
# Create a variable to save all the BSE price rows
all_price_rows = []
skipped_no_link = 0

# For each post, extract the price rows
for i, rp in enumerate(report_posts):
    if not rp["share_summary_xls"]:
        skipped_no_link += 1
        continue
    rows = parse_share_summary_xls(rp["share_summary_xls"], rp["date"])
    all_price_rows.extend(rows)
    if (i + 1) % 25 == 0:
        log.info("Processed %s/%s report posts -- %s rows collected so far.",
                  i + 1, len(report_posts), len(all_price_rows))
    time.sleep(REQUEST_DELAY_SECONDS)

# Once completed, print the number of rows extracted
print(f"Done. Total price rows collected: {len(all_price_rows)}")
print(f"Report posts with no Share Summary link at all: {skipped_no_link} / {len(report_posts)}")

# If any files failed, report the failed files
if all_price_rows:
    all_columns = set()
    for r in all_price_rows:
        all_columns.update(r.keys())
    unrecognized_cols = [c for c in all_columns if c.startswith("unrecognized_")]
    if unrecognized_cols:
        print(f"\nWARNING: found unrecognized columns across the collected rows: {unrecognized_cols}")
        print("Some Share Summary files may use a different layout than expected -- worth checking a sample.")
    else:
        print("\nNo unrecognized columns found -- all parsed files matched the expected layout.")

2026-08-17 02:10:44,264 [INFO] Processed 25/495 report posts -- 725 rows collected so far.
2026-08-17 02:11:33,936 [INFO] Processed 50/495 report posts -- 1450 rows collected so far.
2026-08-17 02:12:23,569 [INFO] Processed 75/495 report posts -- 2154 rows collected so far.
2026-08-17 02:13:13,197 [INFO] Processed 100/495 report posts -- 2854 rows collected so far.
2026-08-17 02:14:02,894 [INFO] Processed 125/495 report posts -- 3554 rows collected so far.
2026-08-17 02:14:52,521 [INFO] Processed 150/495 report posts -- 4254 rows collected so far.
2026-08-17 02:15:42,143 [INFO] Processed 175/495 report posts -- 4954 rows collected so far.
2026-08-17 02:16:31,828 [INFO] Processed 200/495 report posts -- 5654 rows collected so far.
2026-08-17 02:17:21,468 [INFO] Processed 225/495 report posts -- 6354 rows collected so far.
2026-08-17 02:18:11,153 [INFO] Processed 250/495 report posts -- 7406 rows collected so far.
2026-08-17 02:19:00,841 [INFO] Processed 275/495 report posts -- 9206 rows

Done. Total price rows collected: 25046
Report posts with no Share Summary link at all: 0 / 495

No unrecognized columns found -- all parsed files matched the expected layout.


In [25]:
# Save the price data into a DataFrame
price_df = pd.DataFrame(all_price_rows)

# Save the price data into a csv file
price_df.to_csv("bse_price_data.csv", index = False)
print(f"Saved {len(price_df)} rows to bse_price_data.csv")

# Preview the first 10 rows as a sanity check
price_df.head(10)

Saved 25046 rows to bse_price_data.csv


,company_name,last_trade_date,volume,day_high,day_low,last_traded_price,current_close,price_change,bid_price,ask_price,bid_size,ask_size,date
0,Banks Holdings Limited,2026-07-07,0,NaN,NaN,4.5,4.50,0,0.00,4.49,0.0,2750.0,2026-08-14
1,Barbados Dairy Industries Limited,2026-08-06,0,NaN,NaN,2.44,2.44,0,0.00,2.44,0.0,514.0,2026-08-14
2,Barbados Farms Limited,2026-02-18,0,NaN,NaN,0.4,0.40,0,0.00,0.00,0.0,0.0,2026-08-14
3,BICO Limited,2026-05-26,0,NaN,NaN,2,2.00,0,0.00,0.00,0.0,0.0,2026-08-14
4,Cave Shepherd and Company Limited -*,2026-08-06,0,NaN,NaN,9.7,9.70,0,8.50,9.70,30.0,5236.0,2026-08-14
5,CIBC Caribbean Bank Limited,2026-08-13,0,NaN,NaN,2.2,2.20,0,2.15,2.20,23390.0,27238.0,2026-08-14
6,Emera Deposit Receipt -*,2026-08-14,3,25.31,25.31,25.16,25.31,0.15,25.16,25.31,40.0,1183.0,2026-08-14
7,Eppley Caribbean Property Fund SCC - Developme...,2026-06-10,0,NaN,NaN,0.12,0.12,0,0.00,0.12,0.0,24052.0,2026-08-14
8,Eppley Caribbean Property Fund SCC - Value Fund,2026-08-13,0,NaN,NaN,0.51,0.51,0,0.51,0.53,76083.0,27367.0,2026-08-14
9,Goddard Enterprises Limited -*,2026-08-11,0,NaN,NaN,2.65,2.65,0,2.65,2.74,84211.0,10000.0,2026-08-14


## Download PDFs- JSE & BSE

In [1]:
log = logging.getLogger("pdf_downloader")

# Set the browser headers
BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",}

# Define the function to download PDFs from the JSE & BSE websites
def download_pdf(session: requests.Session, url: str, dest_folder: Path, filename: str) -> bool:
 
    dest_folder.mkdir(parents=True, exist_ok=True)
    dest_path = dest_folder / filename

    if dest_path.exists():
        log.info("Already downloaded, skipping: %s", filename)
        return True

    try:
        resp = session.get(url, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to download %s: %s", url, e)
        return False

    if resp.status_code != 200:
        log.warning("%s returned status %s", url, resp.status_code)
        return False

    content_type = resp.headers.get("Content-Type", "")
    if "pdf" not in content_type.lower():
        log.warning("%s wasn't a PDF (Content-Type: %s) -- skipping", url, content_type)
        return False

    dest_path.write_bytes(resp.content)
    return True

### JSE PDF Extraction

In [31]:
# Set the JSE config for PDF download
JSE_BASE_URL = "https://www.jamstockex.com"
JSE_WP_API_POSTS = f"{JSE_BASE_URL}/wp-json/wp/v2/posts"

# Set the JSE session
jse_session = requests.Session()
jse_session.headers.update({**BROWSER_HEADERS, "Referer": f"{JSE_BASE_URL}/"})

# Set the download path
jse_corpus_path = Path("jse_corpus.json")

# Download the PDF for each collected URL
with open(jse_corpus_path, "r", encoding = "utf-8") as f:
    jse_records = json.load(f)
print(f"Loaded {len(jse_records)} records from {jse_corpus_path}")

# Print the number of candidate posts 
jse_candidate_posts = [r for r in jse_records if r["id"].startswith("wp-")]

print(f"{len(jse_candidate_posts)} records have a refetchable WP post ID "
      f"({len(jse_records) - len(jse_candidate_posts)} skipped -- no post ID available).")

Loaded 5531 records from jse_corpus.json
5531 records have a refetchable WP post ID (0 skipped -- no post ID available).


In [32]:
# Define a function to fetch a single post by its numeric ID via the WordPress REST AP
def get_post_content_html(base_url: str, session: requests.Session, post_id: int) -> str:
    url = f"{base_url}/wp-json/wp/v2/posts/{post_id}"
    try:
        resp = session.get(url, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to refetch post %s: %s", post_id, e)
        return ""
    if resp.status_code != 200:
        log.warning("Failed to refetch post %s: status %s", post_id, resp.status_code)
        return ""
    return resp.json().get("content", {}).get("rendered", "")

# Define a function to retrieve every link in the post body whose URL ends in .pdf
def extract_pdf_links(html: str) -> list:
    soup = BeautifulSoup(html or "", "html.parser")
    return [a["href"] for a in soup.find_all("a", href=True) if a["href"].lower().endswith(".pdf")]

# Set the folder to save the downloaded PDFs
jse_pdf_folder = Path("./JSE Data")
downloaded_count = 0
posts_with_pdf = 0
posts_without_pdf = 0

# Save the download progress
progress_path = Path("jse_pdf_link_check_progress.json")
if progress_path.exists():
    with open(progress_path, "r", encoding="utf-8") as f:
        checked_post_ids = set(json.load(f))
    print(f"Resuming -- {len(checked_post_ids)} posts already checked in a previous run.")
else:
    checked_post_ids = set()

# Check every JSE disclosure post for a linked PDF
for i, post in enumerate(jse_candidate_posts):
    post_id_str = post["id"]
    if post_id_str in checked_post_ids:
        continue
    post_id = int(post_id_str.split("-")[1])

    html = get_post_content_html(JSE_BASE_URL, jse_session, post_id)
    pdf_links = extract_pdf_links(html)

    if pdf_links:
        posts_with_pdf += 1
        for pdf_url in pdf_links:
            filename = pdf_url.split("/")[-1]
            if download_pdf(jse_session, pdf_url, jse_pdf_folder, filename):
                downloaded_count += 1
    else:
        posts_without_pdf += 1

    checked_post_ids.add(post_id_str)

    if (i + 1) % 20 == 0:
        with open(progress_path, "w", encoding="utf-8") as f:
            json.dump(list(checked_post_ids), f)
        log.info("Checked %s/%s posts -- %s have a PDF, %s downloaded so far.",
                  i + 1, len(jse_candidate_posts), posts_with_pdf, downloaded_count)
    time.sleep(REQUEST_DELAY_SECONDS)
    
# Final save once every post has been checked,
with open(progress_path, "w", encoding="utf-8") as f:
    json.dump(list(checked_post_ids), f)

# Once completed, print the number of extracted posts
print(f"\nDone. Checked {len(checked_post_ids)} posts.")
print(f"Posts with at least one PDF link: {posts_with_pdf}")
print(f"Posts with no PDF link at all: {posts_without_pdf}")
print(f"Total PDFs downloaded to {jse_pdf_folder.resolve()}: {downloaded_count}")

2026-08-17 02:27:54,122 [INFO] Already downloaded, skipping: Regency-Petroleum-Company-Limited-2nd-Quarter-Report-2026.pdf
2026-08-17 02:28:03,365 [INFO] Checked 20/5531 posts -- 17 have a PDF, 23 downloaded so far.
2026-08-17 02:29:07,125 [INFO] Checked 40/5531 posts -- 35 have a PDF, 49 downloaded so far.
2026-08-17 02:30:04,462 [INFO] Checked 60/5531 posts -- 48 have a PDF, 63 downloaded so far.
2026-08-17 02:31:00,775 [INFO] Checked 80/5531 posts -- 58 have a PDF, 77 downloaded so far.
2026-08-17 02:31:57,329 [INFO] Checked 100/5531 posts -- 70 have a PDF, 96 downloaded so far.
2026-08-17 02:32:51,023 [INFO] Checked 120/5531 posts -- 82 have a PDF, 109 downloaded so far.
2026-08-17 02:33:43,356 [INFO] Checked 140/5531 posts -- 93 have a PDF, 123 downloaded so far.
2026-08-17 02:34:35,253 [INFO] Checked 160/5531 posts -- 101 have a PDF, 134 downloaded so far.
2026-08-17 02:35:25,741 [INFO] Checked 180/5531 posts -- 111 have a PDF, 144 downloaded so far.
2026-08-17 02:36:17,129 [INFO

2026-08-17 03:34:39,239 [INFO] Checked 1500/5531 posts -- 747 have a PDF, 958 downloaded so far.
2026-08-17 03:35:31,492 [INFO] Checked 1520/5531 posts -- 759 have a PDF, 973 downloaded so far.
2026-08-17 03:36:20,492 [INFO] Checked 1540/5531 posts -- 766 have a PDF, 986 downloaded so far.
2026-08-17 03:37:10,420 [INFO] Already downloaded, skipping: MASSY-Notice-of-Interim-Dividend-Declaration.pdf
2026-08-17 03:37:12,723 [INFO] Checked 1560/5531 posts -- 773 have a PDF, 1001 downloaded so far.
2026-08-17 03:38:02,337 [INFO] Checked 1580/5531 posts -- 780 have a PDF, 1008 downloaded so far.
2026-08-17 03:38:53,862 [INFO] Checked 1600/5531 posts -- 791 have a PDF, 1022 downloaded so far.
2026-08-17 03:39:44,068 [INFO] Checked 1620/5531 posts -- 800 have a PDF, 1031 downloaded so far.
2026-08-17 03:40:38,185 [INFO] Checked 1640/5531 posts -- 809 have a PDF, 1042 downloaded so far.
2026-08-17 03:41:30,224 [INFO] Checked 1660/5531 posts -- 819 have a PDF, 1055 downloaded so far.
2026-08-17 

2026-08-17 04:32:15,865 [INFO] Checked 2840/5531 posts -- 1362 have a PDF, 1757 downloaded so far.
2026-08-17 04:33:05,825 [INFO] Checked 2860/5531 posts -- 1373 have a PDF, 1773 downloaded so far.
2026-08-17 04:34:01,866 [INFO] Checked 2880/5531 posts -- 1384 have a PDF, 1793 downloaded so far.
2026-08-17 04:34:56,847 [INFO] Checked 2900/5531 posts -- 1394 have a PDF, 1812 downloaded so far.
2026-08-17 04:35:50,687 [INFO] Checked 2920/5531 posts -- 1405 have a PDF, 1827 downloaded so far.
2026-08-17 04:36:42,656 [INFO] Checked 2940/5531 posts -- 1414 have a PDF, 1839 downloaded so far.
2026-08-17 04:37:33,724 [INFO] Checked 2960/5531 posts -- 1419 have a PDF, 1844 downloaded so far.
2026-08-17 04:38:23,645 [INFO] Checked 2980/5531 posts -- 1425 have a PDF, 1854 downloaded so far.
2026-08-17 04:39:15,490 [INFO] Checked 3000/5531 posts -- 1435 have a PDF, 1865 downloaded so far.
2026-08-17 04:39:30,048 [INFO] Already downloaded, skipping: MANAGEMENT-DISCUSSION-AND-ANALYSIS.pdf
2026-08-1

2026-08-17 05:27:07,961 [INFO] Checked 4060/5531 posts -- 1935 have a PDF, 2505 downloaded so far.
2026-08-17 05:27:15,638 [INFO] Already downloaded, skipping: Chairmans-Update-to-Stockholders-of-Supreme-Ventures-Limited.pdf
2026-08-17 05:27:59,133 [INFO] Checked 4080/5531 posts -- 1949 have a PDF, 2528 downloaded so far.
2026-08-17 05:28:51,822 [INFO] Already downloaded, skipping: 2024-December-Top-Ten-10-Stockholders.pdf
2026-08-17 05:28:51,836 [INFO] Checked 4100/5531 posts -- 1963 have a PDF, 2562 downloaded so far.
2026-08-17 05:29:42,578 [INFO] Checked 4120/5531 posts -- 1967 have a PDF, 2569 downloaded so far.
2026-08-17 05:30:31,151 [INFO] Checked 4140/5531 posts -- 1971 have a PDF, 2576 downloaded so far.
2026-08-17 05:31:23,585 [INFO] Checked 4160/5531 posts -- 1978 have a PDF, 2583 downloaded so far.
2026-08-17 05:32:17,913 [INFO] Checked 4180/5531 posts -- 1986 have a PDF, 2594 downloaded so far.
2026-08-17 05:33:09,276 [INFO] Checked 4200/5531 posts -- 1996 have a PDF, 260

2026-08-17 06:19:48,858 [INFO] Already downloaded, skipping: FESCO_Future-Energy-Source-Co-Ltd_Annual-Report-2024_Final.pdf
2026-08-17 06:19:48,865 [INFO] Already downloaded, skipping: FESCO_Notice-of-2024-Annual-General-Meeting.pdf
2026-08-17 06:19:48,871 [INFO] Already downloaded, skipping: FESCO_2024-Form-of-Proxy.pdf
2026-08-17 06:20:02,505 [INFO] Already downloaded, skipping: JSEs-Monthly-Regulatory-Report-May-2024.pdf
2026-08-17 06:20:05,946 [INFO] Already downloaded, skipping: ANNUALREPORT2024.DIGITALVERSION.pdf
2026-08-17 06:20:06,679 [INFO] Already downloaded, skipping: NOTICE-OF-MEETING-2024-4.pdf
2026-08-17 06:20:07,315 [INFO] Already downloaded, skipping: RADIO-JAMAICA-LIMITED-Form-of-Proxy-3.pdf
2026-08-17 06:20:07,957 [INFO] Already downloaded, skipping: ANNOUNCEMENT.CHANGE.AUDITOR.FINAL_-3.pdf
2026-08-17 06:20:23,516 [INFO] Already downloaded, skipping: ISP-2nd-Quarter-Filing-Report-Revised.pdf
2026-08-17 06:20:33,096 [INFO] Already downloaded, skipping: ISP-Annual-Repor


Done. Checked 5531 posts.
Posts with at least one PDF link: 2561
Posts with no PDF link at all: 2970
Total PDFs downloaded to /home/areas001/Final Project/Final Notebooks/JSE Data: 3401


### BSE PDF Extraction

In [33]:
# Set the BSE configurations
BSE_BASE_URL = "https://bse.com.bb"
BSE_LISTED_COMPANIES_URL = f"{BSE_BASE_URL}/markets/listed-companies"
BSE_SHARE_SUMMARY_URL = f"{BSE_BASE_URL}/reports/share-summary"

# Set the BSE session
bse_session = requests.Session()
bse_session.headers.update({**BROWSER_HEADERS, "Referer": f"{BSE_BASE_URL}/"})

# Retrieve the list of active BSE companies
def get_active_companies() -> list:
    try:
        resp = bse_session.get(BSE_LISTED_COMPANIES_URL, params={"filter": "all", "mode": "filter"}, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to fetch listed companies page: %s", e)
        return []
    if resp.status_code != 200:
        log.warning("Listed companies page returned status %s", resp.status_code)
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    companies = []
    in_active_section = False

    for el in soup.find_all(["h1", "h2", "h3", "h4", "table"]):
        if el.name in ("h1", "h2", "h3", "h4"):
            heading_text = el.get_text(strip=True).lower()
            if "active" in heading_text:
                in_active_section = True
                continue
            elif "delisted" in heading_text:
                in_active_section = False
                continue
        elif el.name == "table" and in_active_section:
            for row in el.find_all("tr"):
                cols = row.find_all(["td", "th"])
                if len(cols) < 2:
                    continue
                name_link = cols[0].find("a", href=True)
                if not name_link:
                    continue
                href = name_link["href"]
                match = re.search(r"instrumentCode=([^&]+)", href)
                if not match:
                    continue
                companies.append({
                    "name": name_link.get_text(strip=True),
                    "instrument_code": match.group(1),
                })

    return companies

# Save the BSE active companies
bse_active_companies = get_active_companies()

# Print the number of companies found
print(f"Found {len(bse_active_companies)} active BSE companies.")

Found 26 active BSE companies.


In [34]:
# Define a function to extract the link to each company's financial documents
def get_company_financial_links(instrument_code: str) -> list:
    try:
        resp = bse_session.get(BSE_SHARE_SUMMARY_URL, params={"instrumentCode": instrument_code}, timeout=TIMEOUT)
    except requests.RequestException as e:
        log.warning("Failed to fetch share summary for %s: %s", instrument_code, e)
        return []
    if resp.status_code != 200:
        log.warning("Share summary for %s returned status %s", instrument_code, resp.status_code)
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    heading = None
    for tag in soup.find_all(["h2", "h3", "h4"]):
        if "financial statements" in tag.get_text(strip=True).lower():
            heading = tag
            break
    if heading is None:
        return []

    table = heading.find_next("table")
    if table is None:
        return []

    links = []
    current_year = None
    for row in table.find_all("tr"):
        cells_ = row.find_all(["td", "th"])
        if not cells_:
            continue
        first_text = cells_[0].get_text(strip=True)
        if re.match(r"^\d{4}$", first_text):
            current_year = first_text
        for cell in cells_:
            a = cell.find("a", href=True)
            if a and a["href"].lower().endswith(".pdf"):
                links.append({
                    "instrument_code": instrument_code,
                    "year": current_year,
                    "doc_type": a.get_text(strip=True),
                    "url": a["href"],
                })
    return links

# Create a variable to save all the BSE links
bse_all_links = []

# For each company find the financial documents URL
for i, company in enumerate(bse_active_companies):
    links = get_company_financial_links(company["instrument_code"])
    bse_all_links.extend(links)
    if (i + 1) % 10 == 0:
        log.info("Processed %s/%s companies -- %s document links found so far.",
                  i + 1, len(bse_active_companies), len(bse_all_links))
    time.sleep(REQUEST_DELAY_SECONDS)

# Print the total number of links found
print(f"Total BSE financial document links found: {len(bse_all_links)}")

2026-08-17 06:30:53,081 [INFO] Processed 10/26 companies -- 98 document links found so far.
2026-08-17 06:31:45,562 [INFO] Processed 20/26 companies -- 148 document links found so far.


Total BSE financial document links found: 148


In [35]:
# Set the BSE PDF download path
bse_pdf_folder = Path("./BSE Data")
bse_downloaded_count = 0

# For each BSE link, download the corresponding PDF
for i, link in enumerate(bse_all_links):
    filename = f"{link['instrument_code']}_{link['year']}_{link['doc_type']}.pdf".replace(" ", "_").replace("/", "-")
    if download_pdf(bse_session, link["url"], bse_pdf_folder, filename):
        bse_downloaded_count += 1

    if (i + 1) % 20 == 0:
        log.info("Processed %s/%s links -- %s PDFs downloaded so far.",
                  i + 1, len(bse_all_links), bse_downloaded_count)
    time.sleep(REQUEST_DELAY_SECONDS)

# Once completed, print the number of files downloaded
print(f"\nDone. Downloaded {bse_downloaded_count} BSE PDFs to {bse_pdf_folder.resolve()}")

2026-08-17 06:33:08,563 [INFO] Processed 20/148 links -- 20 PDFs downloaded so far.
2026-08-17 06:34:04,876 [INFO] Processed 40/148 links -- 40 PDFs downloaded so far.
2026-08-17 06:34:58,402 [INFO] Processed 60/148 links -- 60 PDFs downloaded so far.
2026-08-17 06:35:59,269 [INFO] Processed 80/148 links -- 80 PDFs downloaded so far.
2026-08-17 06:36:55,854 [INFO] Processed 100/148 links -- 100 PDFs downloaded so far.
2026-08-17 06:37:56,094 [INFO] Processed 120/148 links -- 120 PDFs downloaded so far.
2026-08-17 06:38:43,677 [INFO] Processed 140/148 links -- 140 PDFs downloaded so far.



Done. Downloaded 148 BSE PDFs to /home/areas001/Final Project/Final Notebooks/BSE Data


## PDF Text Extraction and Cloud OCR

In [1]:
# Set the PDF paths for each exchange
SOURCES = {
    "jse": Path("./JSE Data"),
    "bse": Path("./BSE Data"),
    "ttse": Path("./TTSE Data/TTSE Annual Reports"),}

TTSE_TRADE_REPORTS_FOLDER = Path("./TTSE Data/TTSE Trade Reports")

pdf_files_by_source = {}

# Create a list of PDF files for each configured source
for source_name, folder in SOURCES.items():
    if not folder.exists():
        print(f"{source_name}: folder {folder} not found -- skipping.")
        pdf_files_by_source[source_name] = []
        continue
    files = sorted(folder.glob("*.pdf"))
    pdf_files_by_source[source_name] = files
    print(f"{source_name}: found {len(files)} PDF(s) in {folder.resolve()}")

if TTSE_TRADE_REPORTS_FOLDER.exists():
    ttse_trade_files = sorted(TTSE_TRADE_REPORTS_FOLDER.glob("*.pdf"))
    print(f"ttse_trade_reports: found {len(ttse_trade_files)} PDF(s) in {TTSE_TRADE_REPORTS_FOLDER.resolve()}")
else:
    ttse_trade_files = []
    print(f"ttse_trade_reports: folder {TTSE_TRADE_REPORTS_FOLDER} not found -- skipping.")

jse: found 3287 PDF(s) in /home/areas001/Final Project/Final Notebooks/JSE Data
bse: found 148 PDF(s) in /home/areas001/Final Project/Final Notebooks/BSE Data
ttse: found 1305 PDF(s) in /home/areas001/Final Project/Final Notebooks/TTSE Data/TTSE Annual Reports
ttse_trade_reports: found 2472 PDF(s) in /home/areas001/Final Project/Final Notebooks/TTSE Data/TTSE Trade Reports


In [3]:
# Import the required packages
import pdfplumber
import fitz

# Define a function to attempt a text direct extraction of the PDFs
def try_direct_extraction(pdf_path: Path) -> str:

    try:
        with pdfplumber.open(pdf_path) as pdf:
            text = "\n".join((page.extract_text() or "") for page in pdf.pages).strip()
        if text:
            return text
    except Exception as e:
        print(f"  pdfplumber failed on {pdf_path.name}: {e}")

    try:
        doc = fitz.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc).strip()
        if text:
            return text
    except Exception as e:
        print(f"  PyMuPDF failed on {pdf_path.name}: {e}")

    return ""

In [4]:
# List month names
MONTH_NAMES = "January|February|March|April|May|June|July|August|September|October|November|December"

# Define a function to extract the date from the financial documents
def extract_date_from_text(text: str):
   
    patterns = [
        rf"(?:as at|as of|year ended|period ended|for the (?:year|nine months|six months|three months) ended)\s+(\d{{1,2}})\s+({MONTH_NAMES}),?\s+(\d{{4}})",
        rf"(?:as at|as of|year ended|period ended|for the (?:year|nine months|six months|three months) ended)\s+({MONTH_NAMES})\s+(\d{{1,2}}),?\s+(\d{{4}})",
    ]
    text_sample = text[:3000]
    for pattern in patterns:
        match = re.search(pattern, text_sample, re.IGNORECASE)
        if match:
            groups = match.groups()
            try:
                if groups[0].isdigit():
                    day, month_str, year = groups
                else:
                    month_str, day, year = groups
                return datetime.strptime(f"{day} {month_str} {year}", "%d %B %Y")
            except (ValueError, AttributeError):
                continue
    return None


# Define a function to extract the date from the file name
def extract_date_from_filename(filename: str):
    """Fallback: parse a date out of the filename/title itself."""
    filename = filename.replace("_", "-")

    match = re.search(rf"({MONTH_NAMES}|[A-Za-z]{{3}})[-\s](\d{{1,2}})[-,\s]+(\d{{4}})", filename, re.IGNORECASE)
    if match:
        month_str, day, year = match.groups()
        for fmt in ("%B-%d-%Y", "%b-%d-%Y"):
            try:
                return datetime.strptime(f"{month_str}-{day}-{year}", fmt)
            except ValueError:
                continue

    match = re.search(rf"({MONTH_NAMES})[-\s](\d{{4}})", filename, re.IGNORECASE)
    if match:
        month_str, year = match.groups()
        try:
            return datetime.strptime(f"1-{month_str}-{year}", "%d-%B-%Y")
        except ValueError:
            pass

    match = re.search(r"\b(20\d{2})\b", filename)
    if match:
        return datetime(int(match.group(1)), 1, 1)

    return None

# Define a function to try text-based extraction first, then filename.
def resolve_chunk_date(chunk_text: str, title: str) -> str:
    date_obj = extract_date_from_text(chunk_text) or extract_date_from_filename(title or "")
    return date_obj.date().isoformat() if date_obj else ""

In [5]:
# Sets the PDF corpus names for each exchange
CORPUS_FILES_TO_BACKFILL = [
    "jse_local_pdf_corpus.json",
    "bse_local_pdf_corpus.json",
    "ttse_local_pdf_corpus.json",]

# If there are missing date fields patch them in the existing corpus
for filename in CORPUS_FILES_TO_BACKFILL:
    path = Path(filename)
    if not path.exists():
        print(f"{filename} not found -- skipping.")
        continue

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    backfilled = 0
    for r in records:
        if not r.get("date") and r.get("text"):
            r["date"] = resolve_chunk_date(r["text"], r.get("filename", ""))
            if r["date"]:
                backfilled += 1

    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"{filename}: backfilled dates for {backfilled} of {len(records)} records")

jse_local_pdf_corpus.json: backfilled dates for 2836 of 3287 records
bse_local_pdf_corpus.json: backfilled dates for 146 of 148 records
ttse_local_pdf_corpus.json: backfilled dates for 1180 of 1305 records


In [38]:
# Import the required package
import time as time_module

# Create variables for the records and already completed
records = []
already_done_by_source = {}

# Check if the record was previously extracted
for source_name in pdf_files_by_source:
    out_path = Path(f"{source_name}_local_pdf_corpus.json")
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            existing = json.load(f)
        records.extend(existing)
        already_done_by_source[source_name] = {r["filename"] for r in existing}
        print(f"{source_name}: resuming -- {len(existing)} already processed, will skip these.")
    else:
        already_done_by_source[source_name] = set()
        
# For each unprocessed file, perform direct text extraction
for source_name, files in pdf_files_by_source.items():
    if not files:
        continue
    already_done = already_done_by_source[source_name]
    remaining_files = [f for f in files if f.name not in already_done]
    if not remaining_files:
        print(f"\n=== {source_name.upper()}: all {len(files)} files already processed, skipping ===")
        continue

    print(f"\n=== Processing {source_name.upper()} ({len(remaining_files)} remaining of {len(files)} total) ===")

    for i, pdf_path in enumerate(remaining_files):
        file_start = time_module.time()
        print(f"[{i+1}/{len(remaining_files)}] {pdf_path.name}...", end=" ")

        text = try_direct_extraction(pdf_path)
       
        method = "direct" if text else "needs_ocr_elsewhere"

        elapsed = time_module.time() - file_start
        print(f"{method} ({elapsed:.1f}s)")
        if elapsed > 30:
            print(f"    ^ SLOW FILE -- {pdf_path.name} took {elapsed:.1f}s")

        records.append({
            "id": f"{source_name}-localpdf-{i}",
            "source": source_name,
            "filename": pdf_path.name,
            "filepath": str(pdf_path.resolve()),
            "text": text,
            "date": resolve_chunk_date(text, pdf_path.name),   # <-- add this line
            "retrieved_via": method,
        })

        if (i + 1) % 10 == 0:
            source_records_so_far = [r for r in records if r["source"] == source_name]
            out_path = Path(f"{source_name}_local_pdf_corpus.json")
            out_path.write_text(json.dumps(source_records_so_far, indent=2, ensure_ascii=False), encoding="utf-8")
            print(f"  -- checkpoint saved ({len(source_records_so_far)} {source_name} records so far)")

# Print the total number of records completed
print(f"\nDone. Total records (including resumed): {len(records)}")
for source_name in pdf_files_by_source:
    source_records = [r for r in records if r["source"] == source_name]
    if not source_records:
        continue
    print(f"\n{source_name.upper()}:")
    print(f"  Direct: {sum(1 for r in source_records if r['retrieved_via'] == 'direct')}")
    print(f"  Needs Cloud OCR: {sum(1 for r in source_records if r['retrieved_via'] == 'needs_ocr_elsewhere')}")


=== Processing JSE (3287 remaining of 3287 total) ===
[1/3287] 0725-Public-Notice-AR-AGM-1.pdf... direct (0.0s)
[2/3287] 1-Class-C-Shareholder-Advisory.pdf... direct (0.2s)
[3/3287] 1-SYGNUS-Class-C-Notice-of-Adjourned-Meeting.pdf... direct (0.2s)
[4/3287] 138-Quarterly-Report-December-2025.pdf... direct (1.2s)
[5/3287] 138-Quarterly-Report-June-2025.pdf... direct (1.4s)
[6/3287] 138-Quarterly-Report-June-2026-Final.pdf... direct (1.5s)
[7/3287] 138-Quarterly-Report-March-2025.pdf... direct (1.4s)
[8/3287] 138-Quarterly-Report-March-2026.pdf... direct (1.2s)
[9/3287] 138-Quarterly-Report-September-2025-JSE.pdf... direct (2.2s)
[10/3287] 138-SL-Corporate-Governance-Policy-Manual-February-11-2025-Revised.pdf... direct (4.0s)
  -- checkpoint saved (10 jse records so far)
[11/3287] 138-Student-Living-Annual-Report-September-2025.pdf... direct (10.2s)
[12/3287] 138-Student-Living-Jamaica-2024-Final.pdf... direct (6.5s)
[13/3287] 138-Student-Living-Jamaica-2025-Final.pdf... direct (8.5s)
[1

  -- checkpoint saved (200 jse records so far)
[201/3287] ASBH-Letter-JSE-dd-April-16-2026-Declaration-of-Dividend-1.pdf... direct (0.2s)
[202/3287] ASBH-Letter-JSE-dd-April-16-2026-Delay-in-Submission-of-2025-Annual-Report-1.pdf... direct (0.1s)
[203/3287] ASBH-Letter-JSE-dd-April-8-2026-Consideration-of-Dividend.pdf... direct (0.1s)
[204/3287] ASBH-Letter-JSE-dd-January-12-2026-Consideration-of-Dividend.pdf... direct (0.0s)
[205/3287] ASBH-Letter-JSE-dd-January-19-2026-Declaration-of-Dividend-1.pdf... direct (0.1s)
[206/3287] ASBH-Letter-JSE-dd-July-15-2026-Consideration-of-Dividend.pdf... direct (0.1s)
[207/3287] ASBH-Letter-JSE-dd-July-15-2026-Declaration-of-Dividend.pdf... direct (0.2s)
[208/3287] ASBH-Letter-JSE-dd-July-22-2026-Declaration-of-Ord-Dividend.pdf... direct (0.2s)
[209/3287] ASBH-Letter-JSE-dd-July-8-2026-Consideration-of-Dividend.pdf... direct (0.1s)
[210/3287] ASBH-Letter-JSE-dd-October-10-2025-Consideration-of-Dividend.pdf... direct (0.0s)
  -- checkpoint saved (21

[381/3287] CAC-2000-Ltd-Financial-Stmts-Q3-PE-31-July-2025-Signed-1.pdf... direct (1.6s)
[382/3287] CAC-2000-Q1-Unaudited-Financials-PE-Jan-31-2026-JSE.pdf... direct (0.3s)
[383/3287] CAC-2000-Unaudited-Results-Q1-YE-Oct-31-2025-Signed.pdf... direct (1.9s)
[384/3287] CAC-2024-Annual-Report_Final.pdf... direct (12.6s)
[385/3287] CAC-2024_Form-of-Proxy.pdf... direct (0.1s)
[386/3287] CAC-2024_Notice-of-AGM.pdf... direct (0.1s)
[387/3287] CAC-Annual-Report-2024.pdf... direct (12.9s)
[388/3287] CAC2000-CEO-Report-Q1-PE-January-31-2026-JSE.docx.pdf... direct (0.2s)
[389/3287] CAR-Unaudited_FS_Q3-Ended-30Sept2024.pdf... needs_ocr_elsewhere (0.0s)
[390/3287] CARIBBEAN-CEMENT-COMPANY-LIMITED-CHANGES-TO-BOARD-OF-DIRECTORS.pdf... direct (0.0s)
  -- checkpoint saved (390 jse records so far)
[391/3287] CARIBBEAN-CEMENT-COMPANY-LIMITED-NOTICE-OF-LATE-SUBMISSION-OF-ANNUAL-REPORT.pdf... direct (0.0s)
[392/3287] CAR_REPORT-TO-OUR-STOCKHOLDERS-Q1-2025-.pdf... direct (0.2s)
[393/3287] CAR_REPORT-TO-OUR-

[575/3287] Directors-and-Connected-Parties-GENAC-Dec-2025.pdf... direct (0.1s)
[576/3287] Directors-and-Connected-Parties-Shareholdings-Report-as-at-May-31-2025.pdf... direct (0.2s)
[577/3287] DirectorsHoldingasatAugust22025.pdf... direct (0.4s)
[578/3287] DirectorsHoldingasatJanuary252025-Executed-Copy.pdf... needs_ocr_elsewhere (0.0s)
[579/3287] DirectorsHoldingasatJuly272024.pdf... direct (0.2s)
[580/3287] DirectorsHoldingasatJune302025.pdf... direct (0.2s)
  -- checkpoint saved (580 jse records so far)
[581/3287] DirectorsHoldingasatMay32025.pdf... direct (0.2s)
[582/3287] DirectorsHoldingasatSeptember302025.pdf... direct (0.3s)
[583/3287] Dividend-letter-2024.pdf... needs_ocr_elsewhere (0.0s)
[584/3287] Dividend_000240-1.pdf... needs_ocr_elsewhere (0.0s)
[585/3287] Dolla-Annual-Report-2024-Final-Single-.pdf... direct (16.0s)
[586/3287] Dolla-Financial-Services-Limited-DOLLA-Audited-Financial-Statements-for-the-Year-Ended-December-31-2024.pdf... direct (11.1s)
[587/3287] Dolla-Fina

[664/3287] Everything-Fresh_2025_Annual-Report_web.pdf... direct (70.4s)
    ^ SLOW FILE -- Everything-Fresh_2025_Annual-Report_web.pdf took 70.4s
[665/3287] EverythingFresh_2024_Annual-Report_Final-1.pdf... direct (9.7s)
[666/3287] Evolve-Dolla-Divestment-Press-Release-1.pdf... direct (0.1s)
[667/3287] Explanatory-Statement-1.pdf... direct (4.2s)
[668/3287] Explanatory-Statement.pdf... direct (4.0s)
[669/3287] Express-Catering-Limited-Audited-Financial-Statements-at-May-31-2024-1.pdf... direct (4.6s)
[670/3287] Express-Catering-Limited-Audited-Financial-Statements-at-May-31-2025.pdf... direct (4.6s)
  -- checkpoint saved (670 jse records so far)
[671/3287] Express-Catering-Limited-Q1-2025-Unaudited-Financial-Statements-Dated-August-31_2024-and-Top-10-and-Directors-Shareholdings.pdf... direct (0.8s)
[672/3287] Express-Catering-Limited-Q1-2026-Unaudited-Financial-Statements-Dated-August-31_2025-and-Top-10-and-Directors-Shareholdings.pdf... direct (0.9s)
[673/3287] Express-Catering-Limit

...

[749/1305] LJ Williams Limited Annual Report 2024.pdf... direct (9.2s)
[750/1305] LJ Williams Limited Annual Report 2025.pdf... direct (7.8s)
  -- checkpoint saved (750 ttse records so far)
[751/1305] Letter from MED - Reconstitution of Audit Risk Compiance and Compensation Committee dd 11.06.26.pdf... needs_ocr_elsewhere (0.0s)
[752/1305] Letter from NGL - Reason for Rescheduled Annual Meeting dd 06.02.2026.pdf... direct (0.1s)
[753/1305] Letters from AGL - Response to query on Top Ten Shareholders dd 18.02.2025.pdf... needs_ocr_elsewhere (0.0s)
[754/1305] MPC Caribbean Clean Energy Limited - Annual Report 2024 (Cover Letter) dd 29.04.2025.pdf... needs_ocr_elsewhere (0.0s)
[755/1305] MPC Caribbean Clean Energy Limited - Notice of Annual General Meeting  dd 27.06.2025.pdf... direct (0.6s)
[756/1305] MPC Caribbean Clean Energy Limited - Notice of Annual General Meeting (Press Release) dd 27.06.2025.pdf... direct (0.6s)
[757/1305] MPC Caribbean Clean Energy Limited - Notice of Annual Gen

[813/1305] Massy Holdings Ltd. - Notice of Interim Dividend Payment dd 09.05.2025.pdf... needs_ocr_elsewhere (0.5s)
[814/1305] Massy Holdings Ltd. - Notice of Partial Final Dividend Payment (Cover Letter) dd 20.11.2025.pdf... direct (0.1s)
[815/1305] Massy Holdings Ltd. - Notice of Partial Final Dividend Payment (Press Release) dd 20.11.2025.pdf... needs_ocr_elsewhere (0.5s)
[816/1305] Massy Holdings Ltd. - Notice of Resignation and Appointment of Directors (Cover Letter) dd 12.03.2025.pdf... direct (0.1s)
[817/1305] Massy Holdings Ltd. - Notice of Resignation of Director (Press Release) dd 12.03.2025.pdf... needs_ocr_elsewhere (0.5s)
[818/1305] Massy Holdings Ltd. - Notice of Resignation of Senior Officer (Cover Letter) dd 09.07.2025.pdf... direct (0.1s)
[819/1305] Massy Holdings Ltd. - Notice of Resignation of Senior Officer (Press Release) dd 06.03.2026.pdf... needs_ocr_elsewhere (0.3s)
[820/1305] Massy Holdings Ltd. - Notice of Resignation of Senior Officer (Press Release) dd 09.07

[939/1305] NCB Financial Group Limited- Proxy Form Annual General Meeting 2023.pdf... direct (0.7s)
[940/1305] NIF Mgt. Disc & Analysis 2025.pdf... direct (0.9s)
  -- checkpoint saved (940 ttse records so far)
[941/1305] National Enterprises Limited - Annual Report 2024  dd 28.01.2025.pdf... direct (0.1s)
[942/1305] National Enterprises Limited - Notice of 2023 Annual Report (Cover Letter) dd 29.01.2024.pdf... direct (0.1s)
[943/1305] National Enterprises Limited - Notice of Annual General Meeting (Cover Letter) dd 14.08.2025.pdf... direct (0.1s)
[944/1305] National Enterprises Limited - Notice of Annual General Meeting (Press Release) dd 14.08.2025.pdf... direct (0.2s)
[945/1305] National Enterprises Limited - Notice of Annual General Meeting (Press Release) dd 15.05.2024.pdf... direct (0.2s)
[946/1305] National Enterprises Limited - Notice of Annual General Meeting (Press Release) dd 19.02.2026.pdf... direct (0.3s)
[947/1305] National Enterprises Limited - Notice of Annual General Me

[1064/1305] Prestige Holdings Limited - Notice of 2024 Final Dividend Payment (Press Release) dd 28.03.2025.pdf... direct (0.2s)
[1065/1305] Prestige Holdings Limited - Notice of Annual General Meeting dd 14.04.2025.pdf... direct (0.4s)
[1066/1305] Prestige Holdings Limited - Notice of Annual Meeting 2025 (Cover Letter) dd 15.04.2026.pdf... direct (0.1s)
[1067/1305] Prestige Holdings Limited - Notice of Annual Meeting 2025 (Press Release) dd 15.04.2026.pdf... direct (0.4s)
[1068/1305] Prestige Holdings Limited - Notice of Appointment of Company Secretary (Cover Letter) dd 26.06.2024.pdf... direct (0.1s)
[1069/1305] Prestige Holdings Limited - Notice of Appointment of Company Secretary (Press release) dd 26.06.2024.pdf... direct (0.3s)
[1070/1305] Prestige Holdings Limited - Notice of Appointment of a Senior Officer (Cover letter)  dd 09.05.2024.pdf... direct (0.1s)
  -- checkpoint saved (1070 ttse records so far)
[1071/1305] Prestige Holdings Limited - Notice of Appointment of a Senior

[1186/1305] Trinidad Cement Limited - Notice of Availability of Annual Report 2025 (Cover Letter) dd 30.04.2026.pdf... needs_ocr_elsewhere (0.7s)
[1187/1305] Trinidad Cement Limited - Notice of Availability of Annual Report 2025 (Press Release) dd 30.04.2026.pdf... needs_ocr_elsewhere (0.7s)
[1188/1305] Trinidad Cement Limited - Notice of Change in Credit Arrangements (Cover Letter) dd 03.02.2026.pdf... needs_ocr_elsewhere (0.0s)
[1189/1305] Trinidad Cement Limited - Notice of Change in Credit Arrangements (Cover Letter) dd 30.06.2026.pdf... needs_ocr_elsewhere (0.0s)
[1190/1305] Trinidad Cement Limited - Notice of Change in Credit Arrangements (Press Release) dd 04.02.2026.pdf... needs_ocr_elsewhere (1.0s)
  -- checkpoint saved (1190 ttse records so far)
[1191/1305] Trinidad Cement Limited - Notice of Change in Credit Arrangements (Press Release) dd 30.06.2026.pdf... needs_ocr_elsewhere (0.7s)
[1192/1305] Trinidad Cement Limited - Notice of Change in Director (Cover Letter) dd 30.07.2

In [39]:
# Save the data for each exchange in a separate file
for source_name in pdf_files_by_source:
    source_records = [r for r in records if r["source"] == source_name]
    if not source_records:
        continue
    out_path = Path(f"{source_name}_local_pdf_corpus.json")
    out_path.write_text(json.dumps(source_records, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved {len(source_records)} records to {out_path.resolve()}")

Saved 3287 records to /home/areas001/Final Project/Final Notebooks/jse_local_pdf_corpus.json
Saved 148 records to /home/areas001/Final Project/Final Notebooks/bse_local_pdf_corpus.json
Saved 1305 records to /home/areas001/Final Project/Final Notebooks/ttse_local_pdf_corpus.json


## TTSE Price Data Extraction

In [40]:
# Define each type of TTSE price file
REPORT_TYPE_PREFIXES = {
    "dcbs": "corporate_bond",
    "dgbs": "government_bond",
    "divs": "index_summary",
    "ds": "daily_quotes",}

# Define a function to classify each file by their name
def classify_report(filename: str):
    stem = filename.lower()
    for prefix, report_type in REPORT_TYPE_PREFIXES.items():
        if stem.startswith(prefix):
            return report_type
    return None

# Set the classification groups
classified = {"corporate_bond": [], "government_bond": [], "index_summary": [], "daily_quotes": [], "unknown": []}

# For each TTSE trade file classify them into type
for p in ttse_trade_files:
    report_type = classify_report(p.name)
    classified[report_type or "unknown"].append(p)

# For each type, print the number of files
for report_type, files in classified.items():
    print(f"{report_type}: {len(files)} file(s)")

# If the file does not fall into an above classification, classify them as unknown
if classified["unknown"]:
    print("\nUnknown-prefix files:")
    for p in classified["unknown"][:10]:
        print(" -", p.name)

corporate_bond: 617 file(s)
government_bond: 617 file(s)
index_summary: 619 file(s)
daily_quotes: 619 file(s)
unknown: 0 file(s)


In [41]:
# Map the bond column names to the file names
BOND_HEADER_MAP = {
    "symbol": "symbol", "security": "security", "issue price": "issue_price",
    "opening quotation": "opening_quotation", "high": "day_high", "low": "day_low",
    "bid price": "bid_price", "bid volume": "bid_volume", "offer price": "offer_price",
    "offer volume": "offer_volume", "price": "last_sale_price", "date": "last_sale_date",
    "last trade yield": "last_trade_yield", "face value traded": "face_value_traded",
    "trade value": "trade_value", "closing quotation": "closing_quotation",
    "change from opening quotation": "change_from_opening",}

# Define a function to find the bond header row
def find_bond_header_row(table: list, max_scan_rows: int = 5):
    for i in range(min(max_scan_rows, len(table))):
        row_values = [str(c).strip().lower() if c else "" for c in table[i]]
        matches = sum(1 for v in row_values if v in BOND_HEADER_MAP)
        if matches >= 3:
            return i
    return None

# Define a function to extract the bond table from the PDFs
def extract_bond_table(pdf_path: Path, bond_type: str) -> list:
    rows_out = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables():
                    if not table:
                        continue
                    header_idx = find_bond_header_row(table)
                    if header_idx is None:
                        continue
                    header_row = [str(c).strip().lower() if c else "" for c in table[header_idx]]
                    column_map = [BOND_HEADER_MAP.get(h, f"unrecognized_{h or 'col'}") for h in header_row]
                    for data_row in table[header_idx + 1:]:
                        if not data_row or not data_row[0]:
                            continue
                        row_dict = dict(zip(column_map, [str(c).strip() if c else "" for c in data_row]))
                        row_dict["bond_type"] = bond_type
                        row_dict["source_file"] = pdf_path.name
                        rows_out.append(row_dict)
    except Exception as e:
        print(f"  Failed to extract bond table from {pdf_path.name}: {e}")
    return rows_out

In [42]:
# Create a variable to save all extracted price rows
all_bond_rows = []

# For each bond file, extract the price rows
for report_type, bond_files in [("corporate", classified["corporate_bond"]),
                                  ("government", classified["government_bond"])]:
    for p in bond_files:
        all_bond_rows.extend(extract_bond_table(p, report_type))

# Print the number of rows extracted
print(f"Total bond rows extracted: {len(all_bond_rows)}")

# Save the bond prices in a DataFrame
bond_df = pd.DataFrame(all_bond_rows)

# Save the bond prices to csv
bond_df.to_csv("ttse_bond_data.csv", index=False)
print(f"Saved {len(bond_df)} rows to ttse_bond_data.csv")

# Preview the top 5 rows of the bond prices
bond_df.head(5)

Total bond rows extracted: 8165
Saved 8165 rows to ttse_bond_data.csv


,symbol,security,issue_price,unrecognized_opening\nquotation,unrecognized_trade prices,unrecognized_col,unrecognized_outstanding,unrecognized_last sale,last_trade_yield,face_value_traded,trade_value,unrecognized_closing\nquotation,change_from_opening,bond_type,source_file
0,NIF090830,"$1,600M NIF SERIES B 5.70% 09.08.2030",100.00,103.37,,28 Dec 2023,103.05,103.37,5.09,,,103.37,,corporate,dcbs-2024-01-02.pdf
1,NIF090838,"$1,200M NIF SERIES C 6.60% 09.08.2038",100.00,106.10,,20 Nov 2023,106.10,106.10,5.97,,,106.10,,corporate,dcbs-2024-01-02.pdf
2,NIF090830,"$1,600M NIF SERIES B 5.70% 09.08.2030",100.00,103.37,,28 Dec 2023,103.05,103.37,5.09,,,103.37,,corporate,dcbs-2024-01-03.pdf
3,NIF090838,"$1,200M NIF SERIES C 6.60% 09.08.2038",100.00,106.10,,20 Nov 2023,106.10,106.10,5.97,,,106.10,,corporate,dcbs-2024-01-03.pdf
4,NIF090830,"$1,600M NIF SERIES B 5.70% 09.08.2030",100.00,103.37,,28 Dec 2023,103.05,103.37,5.09,,,103.37,,corporate,dcbs-2024-01-04.pdf


In [43]:
# Maps the daily trades table headers to the extract headers
SECTOR_TABLE_HEADER = {"security", "issued share capital", "capitalisation value", "trades",
                        "traded volume", "traded value", "index", "change", "change %", "change%"}

TOP_LEVEL_HEADER = {"capitalisation value", "trades", "traded volume", "traded value", "index",
                    "change", "change %", "change%"}


# Define a function to classify the daily index reports
def classify_divs_table(table: list):
    for row in table[:2]:
        row_values = {str(c).strip().lower() if c else "" for c in row}
        if "security" in row_values and len(row_values & SECTOR_TABLE_HEADER) >= 4:
            return "sector"
        if len(row_values & TOP_LEVEL_HEADER) >= 4:
            return "top_level"
    return None

# Defines a function to extract the rows from the daily index reports
def extract_divs_tables(pdf_path: Path) -> dict:
    top_level_rows, sector_rows = [], []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                for table in page.extract_tables():
                    if not table or len(table) < 2:
                        continue
                    table_kind = classify_divs_table(table)
                    if table_kind is None:
                        continue
                    header_row = [str(c).strip().lower() if c else "" for c in table[0]]
                    header_row = [h if h else f"col_{i}" for i, h in enumerate(header_row)]
                    for data_row in table[1:]:
                        if not data_row or not any(data_row):
                            continue
                        row_dict = dict(zip(header_row, [str(c).strip() if c else "" for c in data_row]))
                        row_dict["source_file"] = pdf_path.name
                        (top_level_rows if table_kind == "top_level" else sector_rows).append(row_dict)
    except Exception as e:
        print(f"  Failed to extract DIVS tables from {pdf_path.name}: {e}")
    return {"top_level": top_level_rows, "sector_detail": sector_rows}

In [44]:
all_top_level_rows, all_sector_rows = [], []

# For all index reports, extract the price rows
for p in classified["index_summary"]:
    result = extract_divs_tables(p)
    all_top_level_rows.extend(result["top_level"])
    all_sector_rows.extend(result["sector_detail"])

# Print the number of rows extracted
print(f"Total top-level summary rows: {len(all_top_level_rows)}")
print(f"Total sector detail rows: {len(all_sector_rows)}")

# Save the index price rows to csv
pd.DataFrame(all_top_level_rows).to_csv("ttse_market_summary.csv", index = False)

# Save the index prices in a DataFrame
sector_df = pd.DataFrame(all_sector_rows)
sector_df.to_csv("ttse_sector_detail.csv", index = False)
print("Saved ttse_market_summary.csv and ttse_sector_detail.csv")

# Retrieve the top 5 rows for sanity check
sector_df.head(5)

Total top-level summary rows: 2970
Total sector detail rows: 30829
Saved ttse_market_summary.csv and ttse_sector_detail.csv


,security,issued share\ncapital,capitalisation\nvalue,trades,traded\nvolume,traded\nvalue,index,change,change\n%,source_file,traded value,issued\nshare\ncapital,capitalisation value
0,FCGFH,"251,353,562","12,343,973,430",4,87,"4,272.97",,,,divs-2024-01-02.pdf,NaN,NaN,NaN
1,FCI,"1,577,094,570","11,102,745,773",-,-,-,,,,divs-2024-01-02.pdf,NaN,NaN,NaN
2,NCBFG,"2,466,762,828","8,460,996,500",2,"8,779","30,078.25",,,,divs-2024-01-02.pdf,NaN,NaN,NaN
3,RFHL,"163,673,762","20,087,680,810",4,27,"3,292.56",,,,divs-2024-01-02.pdf,NaN,NaN,NaN
4,SBTT,"176,343,750","12,285,869,063",2,78,"5,416.80",,,,divs-2024-01-02.pdf,NaN,NaN,NaN


In [45]:
# Map the index headers to the extract headers
DS_TOP_LEVEL_HEADER = {"index value", "index change", "change (%)", "volume traded",
                       "value traded ($)", "no. of trades"}

DS_SECURITY_HEADER = {"security", "open price ($)", "high ($)", "low ($)", "o/s bid ($)",
                      "o/s bid vol.", "o/s offer ($)", "o/s offer vol.", "last sale price ($)",
                      "last sale date", "volume traded", "close price ($)", "change ($)"}

# List the TTSE market segments
KNOWN_CATEGORIES = ["Ordinary Shares", "Preference Shares", "Second Tier Shares", "SME Shares",
                    "Mutual Fund Units", "USD Equity Shares"]

# Define a function to remove the embedded newlines/extra whitespace
def normalize_cell(cell) -> str:
    
    if not cell:
        return ""
    return " ".join(str(cell).replace("\n", " ").split())

# Define a function to classify the index reports
def classify_ds_table(table: list):
    for row in table[:2]:
        row_values = {normalize_cell(c).lower() for c in row}
        if len(row_values & DS_SECURITY_HEADER) >= 4:
            return "security_detail"
        if len(row_values & DS_TOP_LEVEL_HEADER) >= 4:
            return "top_level"
    return None

# Define a function to find the headings row
def find_category_headings(page, tolerance: float = 3.0):
 
    words = page.extract_words()
    if not words:
        return []
    words_sorted = sorted(words, key=lambda w: w["top"])
    lines, current_line, current_top = [], [words_sorted[0]], words_sorted[0]["top"]
    for w in words_sorted[1:]:
        if abs(w["top"] - current_top) <= tolerance:
            current_line.append(w)
        else:
            lines.append(current_line)
            current_line, current_top = [w], w["top"]
    lines.append(current_line)

    headings = []
    for line_words in lines:
        line_top = min(w["top"] for w in line_words)
        line_text = " ".join(w["text"] for w in sorted(line_words, key=lambda x: x["x0"]))
        line_text_norm = " ".join(line_text.split()).lower()
        for cat in KNOWN_CATEGORIES:
            if cat.lower() in line_text_norm:
                headings.append((line_top, cat))
                break
    return sorted(headings)

# Define a function to extract the rows from the trade report
def extract_ds_tables(pdf_path: Path) -> dict:
    
    top_level_rows, security_rows = [], []
    current_category = "Ordinary Shares"

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                headings = find_category_headings(page)
                found_tables = sorted(page.find_tables(), key=lambda t: t.bbox[1])

                for t_obj in found_tables:
                    table = t_obj.extract()
                    if not table or len(table) < 2:
                        continue
                    table_kind = classify_ds_table(table)
                    if table_kind is None:
                        continue

                    header_row = [normalize_cell(c) for c in table[0]]
                    header_row = [h if h else f"col_{i}" for i, h in enumerate(header_row)]

                    table_top = t_obj.bbox[1]
                    for heading_top, cat in headings:
                        if heading_top < table_top:
                            current_category = cat
                        else:
                            break

                    for data_row in table[1:]:
                        if not data_row or not any(data_row):
                            continue
                        row_dict = dict(zip(header_row, [normalize_cell(c) for c in data_row]))
                        row_dict["source_file"] = pdf_path.name
                        if table_kind == "top_level":
                            top_level_rows.append(row_dict)
                        else:
                            row_dict["category"] = current_category
                            security_rows.append(row_dict)
    except Exception as e:
        print(f"  Failed to extract DS tables from {pdf_path.name}: {e}")

    return {"top_level": top_level_rows, "security_detail": security_rows}

In [46]:
all_ds_top_level_rows, all_ds_security_rows = [], []

# For each PDF, extract the daily quote price rows
for p in classified["daily_quotes"]:
    result = extract_ds_tables(p)
    all_ds_top_level_rows.extend(result["top_level"])
    all_ds_security_rows.extend(result["security_detail"])

# Print the number of rows
print(f"Total top-level summary rows: {len(all_ds_top_level_rows)}")
print(f"Total security detail rows: {len(all_ds_security_rows)}")

# Save the daily quote price rows to csv
pd.DataFrame(all_ds_top_level_rows).to_csv("ttse_daily_quotes_market_summary.csv", index = False)

# Save the daily quote prices to a DataFrame
ds_security_df = pd.DataFrame(all_ds_security_rows)

# Save the daily quote rows to csv
ds_security_df.to_csv("ttse_daily_quotes_security_detail.csv", index = False)

print("Saved ttse_daily_quotes_market_summary.csv and ttse_daily_quotes_security_detail.csv")

# Print the number of rows per market segment
if not ds_security_df.empty:
    print("\nRows per category:")
    print(ds_security_df["category"].value_counts(dropna=False))

# Print the first 5 rows as a sanity check
ds_security_df.head(5)

Total top-level summary rows: 4952
Total security detail rows: 16041
Saved ttse_daily_quotes_market_summary.csv and ttse_daily_quotes_security_detail.csv

Rows per category:
category
Ordinary Shares       8944
Mutual Fund Units     2476
SME Shares            1935
Preference Shares     1238
USD Equity Shares      829
Second Tier Shares     619
Name: count, dtype: int64


,col_0,Security,Open Price ($),High ($),Low ($),O/S Bid ($),O/S Bid Vol.,O/S Offer ($),O/S Offer Vol.,Last Sale Price ($),Last Sale Date,Volume Traded,Close Price ($),Change ($),source_file,category
0,,AGL,68.49,,,68.33,93,68.49,502,68.50,29-12-2023,,68.49,,ds-2024-01-02.pdf,Ordinary Shares
1,,AHL,21.95,,,21.95,2,22.24,"1,000",22.00,29-12-2023,,21.95,,ds-2024-01-02.pdf,Ordinary Shares
2,,AMBL,45.10,43.00,43.00,43.00,143,45.00,500,43.00,02-01-2024,88,45.10,,ds-2024-01-02.pdf,Ordinary Shares
3,,AMCL,54.06,55.90,55.90,55.00,"2,687",55.90,172,55.90,02-01-2024,33,54.06,,ds-2024-01-02.pdf,Ordinary Shares
4,,FCGFH,49.11,49.15,49.11,49.11,"1,383",49.20,525,49.11,02-01-2024,87,49.11,,ds-2024-01-02.pdf,Ordinary Shares


## Cloud OCR Extraction

In [5]:
# Import required package
from PIL import Image

# Se OCR key (obtained from OCR API website)
API_KEY = "K81492402488957" 

# Set OCR URL
OCR_SPACE_URL = "https://api.ocr.space/parse/image"

# Set OCR extraction configuration
MAX_REQUEST_BYTES = 1_000_000
REQUEST_DELAY_SECONDS = 20
OCR_TIMEOUT = 120

# Set the exchange corpus file paths
CORPUS_FILES = {
    "jse": Path("jse_local_pdf_corpus.json"),
    "bse": Path("bse_local_pdf_corpus.json"),
    "ttse": Path("ttse_local_pdf_corpus.json"),}

In [19]:
# Set the corpora and flagged records variables
corpora = {}
flagged_records = {}

# Retrieve the PDFs that require OCR extraction
for source_name, path in CORPUS_FILES.items():
    if not path.exists():
        print(f"Could not find {path} -- skipping.")
        corpora[source_name] = []
        flagged_records[source_name] = []
        continue
    with open(path, "r", encoding="utf-8") as f:
        corpora[source_name] = json.load(f)
    flagged_records[source_name] = [r for r in corpora[source_name] if r["retrieved_via"] == "needs_ocr_elsewhere"]
    print(f"{source_name}: {len(corpora[source_name])} total, {len(flagged_records[source_name])} flagged for cloud OCR")

jse: 3287 total, 290 flagged for cloud OCR
bse: 148 total, 25 flagged for cloud OCR
ttse: 1305 total, 431 flagged for cloud OCR


In [21]:
# Import the required packages
import fitz
import time as time_module

# Define a function to render each PDF page for OCR extraction
def render_page_compressed(page, max_bytes: int = None, dpi_override: int = None) -> bytes:
    
    max_bytes = max_bytes or MAX_REQUEST_BYTES
    dpi_sequence = [dpi_override] if dpi_override else [200, 150, 100, 75]

    for dpi in dpi_sequence:
        zoom = dpi / 72
        matrix = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix = matrix)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        for quality in (85, 70, 55, 40):
            buf = io.BytesIO()
            img.save(buf, format = "JPEG", quality = quality)
            if buf.tell() <= max_bytes:
                return buf.getvalue()
    return buf.getvalue()

In [22]:
# Define a function to extract each rendered PDF page
def ocr_space_extract_page(page, filename_hint: str, max_attempts: int = 3) -> tuple:
   
    dpi_by_attempt = [200, 120, 75]

    for attempt in range(max_attempts):
        dpi = dpi_by_attempt[min(attempt, len(dpi_by_attempt) - 1)]
        image_bytes = render_page_compressed(page, dpi_override = dpi)

        files = {"file": (filename_hint, image_bytes, "image/jpeg")}
        data = {"apikey": API_KEY, "language": "eng", "isOverlayRequired": "false", "OCREngine": "2"}

        try:
            resp = requests.post(OCR_SPACE_URL, files = files, data = data, timeout = OCR_TIMEOUT)
        except requests.exceptions.Timeout:
            print(f"    Timeout on {filename_hint} (attempt {attempt+1}/{max_attempts}, dpi = {dpi})")
            time_module.sleep(5)
            continue
        except requests.RequestException as e:
            return "", f"request exception: {e}"

        if resp.status_code == 429:
            try:
                retry_after = resp.json().get("retryAfter", 60)
            except ValueError:
                retry_after = 60
            print(f"    Rate limited -- sleeping {retry_after}s...")
            time_module.sleep(retry_after + 5)
            continue

        if resp.status_code == 502 or resp.status_code >= 500:
            print(f"    Server error ({resp.status_code}) on {filename_hint}, attempt {attempt+1}")
            time_module.sleep(8)
            continue

        if resp.status_code != 200:
            return "", f"HTTP {resp.status_code}: {resp.text[:200]}"

        try:
            result = resp.json()
        except ValueError:
            return "", f"non-JSON response: {resp.text[:200]}"

        if result.get("IsErroredOnProcessing"):
            return "", f"API error: {result.get('ErrorMessage')}"

        parsed_results = result.get("ParsedResults") or []
        return (parsed_results[0].get("ParsedText", "").strip(), None) if parsed_results else ("", "empty ParsedResults")

    return "", f"failed after {max_attempts} attempts (timeouts/errors)"

In [23]:
# Define a function to process each extracted PDF page via OCR Cloud
def process_pdf_via_cloud_ocr(pdf_path: Path) -> str:
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        print(f"  Failed to open {pdf_path.name}: {e}")
        return ""

    page_texts = []
    for page_num, page in enumerate(doc):
        try:
            text, error = ocr_space_extract_page(page, filename_hint=f"{pdf_path.stem}_p{page_num+1}.jpg")
            if error:
                print(f"    page {page_num+1} failed: {error}")
            page_texts.append(text)
        except Exception as e:
            print(f"  Failed on {pdf_path.name} page {page_num+1}: {e}")
        time_module.sleep(REQUEST_DELAY_SECONDS)

    return "\n".join(page_texts).strip()

In [26]:
# Set variable to save OCR results
ocr_results = {}

# Set the max number of files per run
MAX_FILES_THIS_RUN = 2000

# Save a progress file for the OCR extraction
partial_results_path = Path("cloud_ocr_partial_results.json")
if partial_results_path.exists():
    with open(partial_results_path, "r", encoding = "utf-8") as f:
        ocr_results = json.load(f)
    print(f"Resuming -- {len(ocr_results)} files already processed in a previous run.")

# For each PDF page, extract via Cloud OCR 
for source_name, recs in flagged_records.items():
    remaining = [r for r in recs if not ocr_results.get(r["filename"])] 
    remaining = remaining[:MAX_FILES_THIS_RUN]
    if not remaining:
        continue
    print(f"\n=== {source_name.upper()}: {len(remaining)} remaining this run (capped at {MAX_FILES_THIS_RUN}) ===")

    for i, record in enumerate(remaining):
        pdf_path = Path(record["filepath"])
        file_start = time_module.time()
        print(f"[{i+1}/{len(remaining)}] {pdf_path.name}...", end=" ")

        text = process_pdf_via_cloud_ocr(pdf_path)
        elapsed = time_module.time() - file_start
        print(f"{'OK' if text else 'FAILED (no text)'} ({elapsed:.1f}s)")

        ocr_results[record["filename"]] = text

        if (i + 1) % 10 == 0:
            with open(partial_results_path, "w", encoding="utf-8") as f:
                json.dump(ocr_results, f, indent = 2, ensure_ascii = False)
            print(f"  -- checkpoint saved ({len(ocr_results)} total results so far)")

# Save the results 
with open(partial_results_path, "w", encoding = "utf-8") as f:
    json.dump(ocr_results, f, indent = 2, ensure_ascii = False)

# Once completed, print the number of sucesses and failures
print(f"\nDone. {len(ocr_results)} files processed via cloud OCR.")
print(f"  Succeeded: {sum(1 for t in ocr_results.values() if t)}")
print(f"  Failed: {sum(1 for t in ocr_results.values() if not t)}")

Resuming -- 20 files already processed in a previous run.

=== JSE: 270 remaining this run (capped at 2000) ===
[1/270] AMGS-TOP-10-AT-31-AUGUST-2024.pdf... OK (21.3s)
[2/270] AMGS-TOP-10-AT-31-MAY-2025.pdf... OK (21.3s)
[3/270] AMGS-TOP-10-AT-31-MAY-2026-1.pdf... OK (21.3s)
[4/270] ASBH-NoAGM-2026.pdf... OK (21.3s)
[5/270] ASBH-Release-Dec-6-2024.pdf... OK (21.3s)
[6/270] AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUST-2024.pdf...     Server error (502) on AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUST-2024_p4.jpg, attempt 1
    Timeout on AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUST-2024_p4.jpg (attempt 2/3, dpi=120)
    Server error (502) on AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUST-2024_p4.jpg, attempt 3
    page 4 failed: failed after 3 attempts (timeouts/errors)
    Server error (502) on AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUST-2024_p5.jpg, attempt 1
    Server error (502) on AUDITED-STATEMENT-OF-COMPREHENSIVE-INCOME-ENDED-AUGUS

[21/270] BaritaInvestmentsLimited-FinancialReport-3Months-December-31-2024-0214-0705.pdf...     page 10 failed: request exception: ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))
    page 11 failed: API error: ['Download Failed']
OK (621.7s)
[22/270] BaritaInvestmentsLimited-FinancialReport-6Months-March-31-2025-v4.pdf... OK (530.9s)
[23/270] CAR-Unaudited_FS_Q3-Ended-30Sept2024.pdf...     Timeout on CAR-Unaudited_FS_Q3-Ended-30Sept2024_p10.jpg (attempt 1/3, dpi=200)
    Server error (502) on CAR-Unaudited_FS_Q3-Ended-30Sept2024_p10.jpg, attempt 2
    Server error (502) on CAR-Unaudited_FS_Q3-Ended-30Sept2024_p10.jpg, attempt 3
    page 10 failed: failed after 3 attempts (timeouts/errors)
OK (505.4s)
[24/270] CCC-Top-Ten-as-at-March-31-2025.pdf...     Server error (502) on CCC-Top-Ten-as-at-March-31-2025_p1.jpg, attempt 1
    Server error (502) on CCC-Top-Ten-as-at-March-31-2025_p1.jpg, attempt 2
    Server error (502) on CCC-Top-Ten-as-at-March-31-2025_p

    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (224.5s)
  -- checkpoint saved (60 total results so far)
[41/270] Directors-Holding-as-at-January-31-2026.pdf... OK (85.2s)
[42/270] Directors-Holding-as-at-June-30-2026.pdf... OK (66.4s)
[43/270] Directors-Holding-as-at-March-31-2026.pdf... OK (63.2s)
[44/270] Directors-Holding-as-at-October-31-2025.pdf... OK (63.2s)
[45/270] Directors-Holdings-as-at-November-1-2025-1.pdf... OK (84.6s)
[46/270] Directors-Share-Purchase-Notice-2-27-Aug-2024.pdf... OK (20.9s)
[47/270] Directors-Share-Purchase-Notice-REVISED-19-22.8.24.pdf... OK (20.9s)
[48/270] DirectorsHoldingasatJanuary252025-Executed-Copy.pdf... OK (85.2s)
[49/270] Dividend-letter-2024.pdf... OK (21.7s)
[50/270] Dividend_000240-1.pdf... OK (21.3s)
  -- checkpoint saved (70 total results so far)
[51/270] EFRESH-Report-to-Shareholders-ended-March-31-2025_000085.pdf... OK (42.9s)
[52/270] EFRESH-Unaudited-Financial-ended-31-March-2025_000084.pdf...     Timeout

    Server error (502) on JSE-Filing-Report-1st-Quarter-ended-March-31-2026_p6.jpg, attempt 2
    Server error (502) on JSE-Filing-Report-1st-Quarter-ended-March-31-2026_p6.jpg, attempt 3
    page 6 failed: failed after 3 attempts (timeouts/errors)
OK (522.0s)
[102/270] JSE-Filing-Report-2nd-Qtr-Ended-June-30-2026.pdf...     Server error (502) on JSE-Filing-Report-2nd-Qtr-Ended-June-30-2026_p10.jpg, attempt 1
    Server error (502) on JSE-Filing-Report-2nd-Qtr-Ended-June-30-2026_p10.jpg, attempt 2
    Server error (502) on JSE-Filing-Report-2nd-Qtr-Ended-June-30-2026_p10.jpg, attempt 3
    page 10 failed: failed after 3 attempts (timeouts/errors)
    Server error (502) on JSE-Filing-Report-2nd-Qtr-Ended-June-30-2026_p11.jpg, attempt 1
OK (574.3s)
[103/270] JSE-Letter-MJE-Annual-Report-31-December-2024-Final.pdf... OK (20.9s)
[104/270] JSE-Sep-2024-Unaudited-report.pdf... OK (214.5s)
[105/270] JSEG-2ND-Quarter-Financial-Statements-June-2025.pdf...     Server error (502) on JSEG-2ND-Quar

    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p2.jpg, attempt 2
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p2.jpg, attempt 3
    page 2 failed: failed after 3 attempts (timeouts/errors)
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p3.jpg, attempt 1
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p3.jpg, attempt 2
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p3.jpg, attempt 3
    page 3 failed: failed after 3 attempts (timeouts/errors)
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p4.jpg, attempt 1
    Server error (503) on Radio-Jamaica-Limited-Unadited-Financial-Statements-as-at-September-2025_p4.jpg, attempt 2
    Server error (503) on Radio-Jamaica-Limited-Unadited-Fi

...

    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (44.5s)
[270/431] Massy Holdings Ltd. - Notice of change in Senior Officer (Press release) dd 12.03.2024.pdf...     Server error (503) on Massy Holdings Ltd. - Notice of change in Senior Officer (Press release) dd 12.03.2024_p1.jpg, attempt 1
    Server error (503) on Massy Holdings Ltd. - Notice of change in Senior Officer (Press release) dd 12.03.2024_p1.jpg, attempt 2
    Server error (503) on Massy Holdings Ltd. - Notice of change in Senior Officer (Press release) dd 12.03.2024_p1.jpg, attempt 3
    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (44.4s)
  -- checkpoint saved (585 total results so far)
[271/431] Massy Holdings Ltd. - Notice of change in Senior Officer (Press release) dd 25.03.2024.pdf... OK (21.1s)
[272/431] Massy Holdings Ltd. -Notice of Appointment of a Director (Press release) dd 14.08.2024.pdf... OK (21.7s)
[273/431] Massy Holdings Ltd. -Notice of change in S

    Server error (503) on National Enterprises Limited - Notice of Third Dividend Payment (Press release) dd 13.12.2023_p1.jpg, attempt 2
    Server error (503) on National Enterprises Limited - Notice of Third Dividend Payment (Press release) dd 13.12.2023_p1.jpg, attempt 3
    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (44.4s)
  -- checkpoint saved (615 total results so far)
[301/431] National Enterprises Limited - Shareholdings Report as at June, 30 2024 dd 06.08.2024.pdf...     Server error (503) on National Enterprises Limited - Shareholdings Report as at June, 30 2024 dd 06.08.2024_p1.jpg, attempt 1
OK (50.8s)
[302/431] National Flour Mills Limited - Amended Notice of Annual General Meeting (Cover Letter) dd 10.12.2025.pdf... OK (22.3s)
[303/431] National Flour Mills Limited - Amended Notice of Annual General Meeting (Press Release) dd 10.12.2025.pdf... OK (22.3s)
[304/431] National Flour Mills Limited - Notice of AGM, Proxy Form and Management Prox

    Server error (503) on One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 02.04.2025_p1.jpg, attempt 2
    Server error (503) on One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 02.04.2025_p1.jpg, attempt 3
    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (44.5s)
[335/431] One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 30.04.2026.pdf...     Server error (503) on One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 30.04.2026_p1.jpg, attempt 1
    Server error (503) on One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 30.04.2026_p1.jpg, attempt 2
    Server error (503) on One Caribbean Media Limited - Notice of Final Dividend Payment (Cover Letter) dd 30.04.2026_p1.jpg, attempt 3
    page 1 failed: failed after 3 attempts (timeouts/errors)
FAILED (no text) (44.5s)
[336/431] One Caribbean Media Limited -

    page 94 failed: failed after 3 attempts (timeouts/errors)
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p95.jpg, attempt 1
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p95.jpg, attempt 2
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p95.jpg, attempt 3
    page 95 failed: failed after 3 attempts (timeouts/errors)
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p118.jpg, attempt 1
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p118.jpg, attempt 2
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p118.jpg, attempt 3
    page 118 failed: failed after 3 attempts (timeouts/errors)
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p119.jpg, attempt 1
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p119.jpg, attempt 2
    Server error (503) on Prestige Holdings Limited Annual Report 2024_p119.jpg, attempt

[413/431] Trinidad and Tobago NGL Limited - Notice of Resignation of a Director (Press release) dd 22.11.2024.pdf... OK (20.9s)
[414/431] Trinidad and Tobago NGL Limited - Notice of Resignation of a Director Express Clippings.pdf... OK (22.5s)
[415/431] Trinidad and Tobago NGL Limited - Notice of Resignation of a Director Guardian Clippings.pdf... OK (21.6s)
[416/431] Trinidad and Tobago NGL Limited - Notice of Special Dividend (Press Release) dd 10.04.2026.pdf... OK (21.2s)
[417/431] Trinidad and Tobago NGL Limited - Notice to Shareholders - Accesibility of Annual Report 29.06.2026.pdf... OK (25.5s)
[418/431] Trinidad and Tobago NGL Limited - Submission of 2023 Annual Report  (Cover letter) dd 02.07.2024.pdf... OK (30.6s)
[419/431] Unilever Caribbean Limited - Annual Report 2024 (Cover Letter) dd 30.04.2025.pdf... OK (77.8s)
[420/431] Unilever Caribbean Limited - Notice of 2nd Interim Dividend Payment (Press Release) dd 14.08.2025.pdf...     Server error (503) on Unilever Caribbean Li

In [27]:
# Merge the Cloud OCR results into the existing exchange corpus
for source_name, path in CORPUS_FILES.items():
    if not path.exists():
        continue
    updated_count = 0
    for record in corpora[source_name]:
        if record["filename"] in ocr_results and ocr_results[record["filename"]]:
            record["text"] = ocr_results[record["filename"]]
            record["retrieved_via"] = "cloud_ocr"
            updated_count += 1
    path.write_text(json.dumps(corpora[source_name], indent = 2, ensure_ascii = False), encoding = "utf-8")
    print(f"{source_name}: updated {updated_count} records, saved to {path.resolve()}")

jse: updated 281 records, saved to /home/areas001/Final Project/Final Notebooks/jse_local_pdf_corpus.json
bse: updated 23 records, saved to /home/areas001/Final Project/Final Notebooks/bse_local_pdf_corpus.json
ttse: updated 336 records, saved to /home/areas001/Final Project/Final Notebooks/ttse_local_pdf_corpus.json


In [28]:
# Print a summary of the text extraction results
for source_name in CORPUS_FILES:
    if not corpora.get(source_name):
        continue
    df = pd.DataFrame(corpora[source_name])
    print(f"\n{source_name.upper()} -- final retrieved_via breakdown:")
    print(df["retrieved_via"].value_counts())
    print(f"Average text length: {df['text'].str.len().mean():.0f} characters")


JSE -- final retrieved_via breakdown:
retrieved_via
direct                 2997
cloud_ocr               281
needs_ocr_elsewhere       9
Name: count, dtype: int64
Average text length: 30574 characters

BSE -- final retrieved_via breakdown:
retrieved_via
direct                 123
cloud_ocr               23
needs_ocr_elsewhere      2
Name: count, dtype: int64
Average text length: 79190 characters

TTSE -- final retrieved_via breakdown:
retrieved_via
direct                 874
cloud_ocr              336
needs_ocr_elsewhere     95
Name: count, dtype: int64
Average text length: 21051 characters
